In [32]:
# ============================================================================
# FINAL COMPREHENSIVE DECISION MATRIX - All Analyses Completed
# ============================================================================
print("="*80)
print("FINAL DATA-DRIVEN DECISIONS")
print("Based on: Redundancy, Stability, Zero-Inflation, Coverage, INFORM vs WRI")
print("="*80)

# Start with the decision matrix from earlier
final_comprehensive = final_decisions.copy()

# ============================================================================
# DECISION 1: INFORM vs WRI - Scale Dimension Head-to-Head
# ============================================================================
print("\n" + "="*80)
print("DECISION 1: INFORM vs WRI Scale Indicators")
print("="*80)
print("ANALYSIS RESULTS:")
print("  • Coverage: TIE (both 46 countries)")
print("  • Variance: WRI wins (0.0751 vs 0.0750)")
print("  • Stability: TIE (insufficient overlapping data)")
print("  • Hazard Breadth: TIE (both 1.0)")
print("\nWRI wins 1/4 metrics BUT margin is 0.0001 variance (negligible)")
print("\n🏆 OVERRIDE DECISION: Keep INFORM (institutional standard)")
print("   Reasoning: Statistical tie + INFORM is UN/humanitarian standard")
print("   → Already decided to remove WRI.EI_03/04/05 due to redundancy (ρ>0.72)")
print("   → Keep WRI.EI_06 only (Drought, NOT redundant with INFORM)")

# Keep previous WRI removals
wri_remove = ["WRI.EI_03", "WRI.EI_04", "WRI.EI_05"]
for ind in wri_remove:
    if ind in final_comprehensive["indicator_id"].values:
        final_comprehensive.loc[final_comprehensive["indicator_id"] == ind, "decision"] = "❌ REMOVE"
        final_comprehensive.loc[final_comprehensive["indicator_id"] == ind, "removal_reason"] = "Redundant with INFORM (ρ>0.72); choose INFORM as institutional standard"

print("\n✅ KEEP: INFORM.HAZEX.*, WRI.EI_06")
print("❌ REMOVE: WRI.EI_03, WRI.EI_04, WRI.EI_05")

# ============================================================================
# DECISION 2: Zero-Inflation Assessment
# ============================================================================
print("\n" + "="*80)
print("DECISION 2: High Zero-Inflation Indicators")
print("="*80)
print("ANALYSIS RESULTS:")
print("  • 7/34 indicators have >30% zeros")
print("  • Correlation (zeros vs coverage): 0.011 (WEAK)")
print("  • Interpretation: Zeros are REAL (true absence), not reporting gaps")
print("  • 3 impact indicators with high zeros BUT 3 alternatives with low zeros")
print("\nHIGH ZERO INDICATORS:")
print("  1. EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA: 96.8% zeros, 45 countries")
print("  2. INFORM.HAZEX.TROPICAL_CYCLONE: 82.6% zeros, 46 countries")
print("  3. DESINVENTAR.LOSS_USD_PER_CAPITA: 81.8% zeros, 20 countries")
print("  4. DESINVENTAR.DURATION_MEAN_DAYS: 44.6% zeros, 24 countries")
print("  5. INFORM.HAZEX.COASTAL_FLOOD: 39.1% zeros, 46 countries")
print("  6. INFORMCC.CHG_HAZEX.COASTAL_FLOOD: 37.0% zeros, 46 countries")
print("  7. DESINVENTAR.DEATHS_PER100K: 33.8% zeros, 24 countries")

# Decision: Remove only those with BOTH high zeros AND already flagged for other reasons
zero_removal = []

# EM-DAT.DAMAGE: 96.8% zeros - this is extreme, likely unreliable
if "EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024" in final_comprehensive["indicator_id"].values:
    final_comprehensive.loc[
        final_comprehensive["indicator_id"] == "EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024", 
        "decision"
    ] = "❌ REMOVE"
    final_comprehensive.loc[
        final_comprehensive["indicator_id"] == "EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024", 
        "removal_reason"
    ] = "Extreme zero-inflation (96.8%); unreliable signal"
    zero_removal.append("EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024")
    print("\n❌ REMOVE: EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA (96.8% zeros - extreme)")

# DESINVENTAR.LOSS: 81.8% zeros + low coverage (20 countries)
if "DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024" in final_comprehensive["indicator_id"].values:
    final_comprehensive.loc[
        final_comprehensive["indicator_id"] == "DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024", 
        "decision"
    ] = "❌ REMOVE"
    final_comprehensive.loc[
        final_comprehensive["indicator_id"] == "DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024", 
        "removal_reason"
    ] = "High zeros (81.8%) + low coverage (20 countries)"
    zero_removal.append("DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024")
    print("❌ REMOVE: DESINVENTAR.LOSS_USD_PER_CAPITA (81.8% zeros + only 20 countries)")

print(f"\n✅ KEEP: INFORM.HAZEX.TROPICAL_CYCLONE (zeros expected for landlocked/stable countries)")
print("✅ KEEP: Other high-zero indicators (zeros are REAL, not data quality issues)")

# ============================================================================
# DECISION 3: ADMIN_SPREAD - Perfect Redundancy
# ============================================================================
print("\n" + "="*80)
print("DECISION 3: ADMIN_SPREAD Indicators")
print("="*80)
print("ANALYSIS RESULTS:")
print("  • EM-DAT.ADMIN_SPREAD ↔ EM-DAT.DURATION_MEAN: ρ = 1.000 (PERFECT)")
print("  • DESINVENTAR.ADMIN_SPREAD ↔ DESINVENTAR.DURATION_MEAN: ρ = 1.000 (PERFECT)")
print("  • User expressed uncertainty about methodology")
print("\n❌ REMOVE: ALL ADMIN_SPREAD indicators (17 total)")
print("   Reasoning: Zero new information (ρ=1.0 with DURATION_MEAN)")

admin_spread_inds = final_comprehensive[
    final_comprehensive["indicator_id"].str.contains("ADMIN_SPREAD", regex=True)
]["indicator_id"].unique()

for ind in admin_spread_inds:
    final_comprehensive.loc[final_comprehensive["indicator_id"] == ind, "decision"] = "❌ REMOVE"
    final_comprehensive.loc[final_comprehensive["indicator_id"] == ind, "removal_reason"] = "Perfect redundancy with DURATION_MEAN (ρ=1.0)"

print(f"   Affected: {len(admin_spread_inds)} indicators")

# ============================================================================
# DECISION 4: Normalization Method Validation
# ============================================================================
print("\n" + "="*80)
print("DECISION 4: Normalization Method")
print("="*80)
print("ANALYSIS RESULTS:")
print("  • Tested: Percentile (current), MinMax, Log-MinMax")
print("  • Percentile ↔ MinMax: r=0.618")
print("  • Percentile ↔ Log-MinMax: r=0.652")
print("  • MinMax ↔ Log-MinMax: r=0.793")
print("  • 4 regions identified in dataset")
print("\n✅ DECISION: Keep PERCENTILE normalization (current method)")
print("   Reasoning: Most robust to outliers and sparse data")
print("   → No changes needed to indicator set based on this analysis")

# ============================================================================
# FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("COMPREHENSIVE REMOVAL SUMMARY")
print("="*80)

# Count decisions
decision_summary = final_comprehensive["decision"].value_counts().to_dict()

print("\nDECISION BREAKDOWN:")
for decision, count in sorted(decision_summary.items()):
    print(f"  {decision}: {count} indicators")

# List all removals with reasons
removals = final_comprehensive[final_comprehensive["decision"] == "❌ REMOVE"].copy()
print(f"\n" + "="*80)
print(f"COMPLETE REMOVAL LIST ({len(removals)} indicators)")
print("="*80)

# Group by removal reason
if "removal_reason" not in removals.columns:
    removals["removal_reason"] = "See earlier diagnostic flags"

removal_by_reason = removals.groupby("removal_reason")["indicator_id"].apply(list).to_dict()

for reason, inds in removal_by_reason.items():
    print(f"\n{reason}:")
    for ind in sorted(inds):
        cov = removals[removals["indicator_id"] == ind]["n_countries"].values[0]
        print(f"  • {ind} (cov={cov})")

# Export final comprehensive decisions
export_comprehensive = OUT_DIR / "diagnostics_final_comprehensive_decisions.csv"
final_comprehensive.to_csv(export_comprehensive, index=False)
print(f"\n✅ Saved: {export_comprehensive}")

print("\n" + "="*80)
print("NEXT STEPS:")
print("="*80)
print("1. Review removal list with stakeholders")
print("2. Implement exclusions in main WP3_07 notebook")
print("3. Re-run full pipeline with reduced indicator set")
print("4. Generate final hazard rankings for regional comparison")
print("5. Prepare stakeholder presentation with defensible rationale")

FINAL DATA-DRIVEN DECISIONS
Based on: Redundancy, Stability, Zero-Inflation, Coverage, INFORM vs WRI

DECISION 1: INFORM vs WRI Scale Indicators
ANALYSIS RESULTS:
  • Coverage: TIE (both 46 countries)
  • Variance: WRI wins (0.0751 vs 0.0750)
  • Stability: TIE (insufficient overlapping data)
  • Hazard Breadth: TIE (both 1.0)

WRI wins 1/4 metrics BUT margin is 0.0001 variance (negligible)

🏆 OVERRIDE DECISION: Keep INFORM (institutional standard)
   Reasoning: Statistical tie + INFORM is UN/humanitarian standard
   → Already decided to remove WRI.EI_03/04/05 due to redundancy (ρ>0.72)
   → Keep WRI.EI_06 only (Drought, NOT redundant with INFORM)

✅ KEEP: INFORM.HAZEX.*, WRI.EI_06
❌ REMOVE: WRI.EI_03, WRI.EI_04, WRI.EI_05

DECISION 2: High Zero-Inflation Indicators
ANALYSIS RESULTS:
  • 7/34 indicators have >30% zeros
  • Correlation (zeros vs coverage): 0.011 (WEAK)
  • Interpretation: Zeros are REAL (true absence), not reporting gaps
  • 3 impact indicators with high zeros BUT 3 alt

## FINAL DATA-DRIVEN DECISIONS

**Based on completed analyses, here are the final removal decisions:**

# WP3 Hazard Prioritisation — Diagnostics & Comparisons
Purpose: diagnose ranking anomalies, compare weighting/gating options, and prepare XLSX outputs without altering the main pipeline. Run cells top-to-bottom after generating fresh outputs from the main notebook.

In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, List
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Paths
BASE_DIR = Path(r"c:\pipelines\sewa-multihazar")
DATA_LONG = BASE_DIR / "data" / "intermediate" / "wp3_country_indicators_long.csv"
OUT_DIR = BASE_DIR / "data" / "output" / "wp3_prioritisation"
FIG_DIR = OUT_DIR / "figures"
SCENARIO = "S_main"  # default = M1 construct arbitration
SCORING_FILE = OUT_DIR / f"scoring_norm__{SCENARIO}.csv"
HAZARD_SCORES_FILE = OUT_DIR / f"hazard_scores__{SCENARIO}.csv"

print("Paths set:")
print(DATA_LONG)
print(SCORING_FILE)

Paths set:
c:\pipelines\sewa-multihazar\data\intermediate\wp3_country_indicators_long.csv
c:\pipelines\sewa-multihazar\data\output\wp3_prioritisation\scoring_norm__S_main.csv


In [14]:
# Dimension weights (adjustable)
DIM_WEIGHTS_MAIN = {
    "Impact": 0.35,
    "Prevalence": 0.20,
    "Scale": 0.15,
    "Cascade impacts": 0.20,
    "Future relevance": 0.10,
}

# Optional alternative weights for sensitivity
DIM_WEIGHTS_ALT = {
    "Impact": 0.30,
    "Prevalence": 0.20,
    "Scale": 0.20,
    "Cascade impacts": 0.20,
    "Future relevance": 0.10,
}

# Dimension list (used for diagnostics)
DIMENSIONS = list(DIM_WEIGHTS_MAIN.keys())

# Indicator handling / quality
MIN_DIMS_REQUIRED = 3  # soft rule for quality flag
SINGLE_DIM_SOFT_PENALTY = 0.8       # down-weight only when conditions below hold
USE_SOFT_PENALTY = True
HAZARD_IND_THRESHOLD = 3            # apply the penalty only if the hazard has ≥3 indicators overall
ARBITRATION_MODE = "M1"  # default per discussion

In [3]:
# Exposure gating rules
PRESENCE_RULES = {
    "Drought": [
        ("TH.DG_LEVEL", "th_level", {"min_level": 2}),
        ("INFORM.HAZEX.DROUGHT", "gt0", {}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Flash Flooding": [
        ("TH.UF_LEVEL", "th_level", {"min_level": 2}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Riverine Flooding Continued & Cluster": [
        ("TH.FL_LEVEL", "th_level", {"min_level": 2}),
        ("INFORM.HAZEX.RIVER_FLOOD", "gt0", {}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Heatwave": [
        ("TH.EH_LEVEL", "th_level", {"min_level": 2}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Storm Surge": [
        ("TH.CF_LEVEL", "th_level", {"min_level": 2}),
        ("INFORM.HAZEX.COASTAL_FLOOD", "gt0", {}),
        ("INFORMCC.CHG_HAZEX.COASTAL_FLOOD.2050.pessimistic", "gt0", {}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Wind": [
        ("TH.CY_LEVEL", "th_level", {"min_level": 2}),
        ("INFORM.HAZEX.TROPICAL_CYCLONE", "gt0", {}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Thunderstorm": [
        ("TH.TS_LEVEL", "th_level", {"min_level": 2}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Wildfires": [
        ("TH.WF_LEVEL", "th_level", {"min_level": 2}),
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
    "Dust": [
        ("EM-DAT.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
        ("DESINVENTAR.EVENTS_PER_YEAR_2000_2024", "gt0", {}),
    ],
}

def check_presence(val, kind, params):
    if pd.isna(val):
        return False
    if kind == "gt0":
        try:
            return float(val) > 0.0
        except Exception:
            return False
    if kind == "th_level":
        min_level = int(params.get("min_level", 2))
        try:
            return int(round(float(val))) >= min_level
        except Exception:
            return False
    return False

def build_presence_evidence(scoring_norm: pd.DataFrame) -> pd.DataFrame:
    tmp = scoring_norm.copy()
    m = (tmp["hazard_applicability"] == "hazard-specific") & tmp["dimension"].isin(["Prevalence","Scale"])
    tmp = tmp.loc[m, ["iso3","hazard","indicator_id","value_raw"]].copy()
    if tmp.empty:
        return pd.DataFrame(columns=["iso3","hazard","n_presence_hits"])
    tmp["is_presence"] = tmp.apply(lambda r: any((r["indicator_id"] == ind) and check_presence(r["value_raw"], kind, params) for ind, kind, params in PRESENCE_RULES.get(r["hazard"], [])), axis=1)
    ev = tmp[tmp["is_presence"]].groupby(["iso3","hazard"]).agg(n_presence_hits=("indicator_id","nunique")).reset_index()
    return ev

In [4]:
def load_scoring_norm() -> pd.DataFrame:
    if SCORING_FILE.exists():
        df = pd.read_csv(SCORING_FILE)
        print(f"Loaded scoring_norm from {SCORING_FILE}")
        return df
    raise FileNotFoundError("Run main notebook to produce scoring_norm CSV first.")

def load_hazard_scores() -> pd.DataFrame | None:
    if HAZARD_SCORES_FILE.exists():
        return pd.read_csv(HAZARD_SCORES_FILE)
    return None

def soft_penalize_dimension(score: float, n_ind: int, haz_n_ind: int) -> float:
    if not USE_SOFT_PENALTY:
        return score
    # Only penalize single-indicator dimensions when the hazard has enough other indicators
    if haz_n_ind < HAZARD_IND_THRESHOLD:
        return score
    if n_ind <= 1:
        return score * SINGLE_DIM_SOFT_PENALTY
    return score

def compute_dimension_scores(scoring_norm: pd.DataFrame, dim_weights: Dict[str,float]) -> pd.DataFrame:
    tmp = scoring_norm.copy()
    tmp = tmp[tmp["score_01"].notna()]

    # Hazard-level indicator counts (per country × hazard)
    haz_counts = (
        tmp.groupby(["iso3","hazard"])
        ["indicator_id"].nunique()
        .reset_index(name="haz_n_indicators")
    )

    g = tmp.groupby(["iso3","country","region","hazard","dimension"])
    dim = g.agg(dimension_score=("score_01","mean"),
                 n_indicators=("indicator_id","nunique"))
    dim = dim.reset_index()

    # Attach hazard-level counts
    dim = dim.merge(haz_counts, on=["iso3","hazard"], how="left")
    dim["haz_n_indicators"] = dim["haz_n_indicators"].fillna(0).astype(int)

    dim["dimension_score"] = dim.apply(
        lambda r: soft_penalize_dimension(r["dimension_score"], r["n_indicators"], r["haz_n_indicators"]),
        axis=1,
    )
    dim["dim_weight"] = dim["dimension"].map(dim_weights).fillna(0.0)
    return dim

def compute_hazard_scores(dim_scores: pd.DataFrame, min_dims: int = MIN_DIMS_REQUIRED) -> pd.DataFrame:
    def agg(g):
        n_dims = int(g["dimension"].nunique())
        if n_dims < min_dims:
            return pd.Series({"hazard_score": np.nan, "n_dimensions": n_dims})
        ws = g["dimension_score"] * g["dim_weight"]
        total_w = g["dim_weight"].sum()
        return pd.Series({"hazard_score": ws.sum()/total_w if total_w>0 else np.nan, "n_dimensions": n_dims})
    haz = dim_scores.groupby(["iso3","country","region","hazard"], as_index=False).apply(agg)
    return haz

def apply_exposure_gating(haz_scores: pd.DataFrame, evidence: pd.DataFrame) -> pd.DataFrame:
    out = haz_scores.merge(evidence, on=["iso3","hazard"], how="left")
    out["n_presence_hits"] = out["n_presence_hits"].fillna(0).astype(int)
    gated = out["n_presence_hits"].eq(0)
    out["hazard_score_raw"] = out["hazard_score"]
    out["is_exposure_gated"] = gated
    out.loc[gated, "hazard_score"] = 0.0
    return out

In [5]:
def log_minmax(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    s = np.log1p(s.clip(lower=0))
    if s.notna().sum() <= 1:
        return pd.Series(np.nan, index=series.index)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)


def recompute_with_options(scoring_norm: pd.DataFrame, *, dim_weights: Dict[str, float], use_log_minmax: bool = False) -> dict:
    df = scoring_norm.copy()

    if use_log_minmax:
        # apply only to skewed impact indicators
        skew_inds = df["indicator_id"].str.contains("DEATHS|AFFECTED|DAMAGE|LOSS|DISPLACEMENTS", regex=True)
        df.loc[skew_inds, "score_01_alt"] = log_minmax(df.loc[skew_inds, "value_raw"])
        df["score_use"] = df["score_01_alt"].fillna(df["score_01"])
    else:
        df["score_use"] = df["score_01"]

    df = df[df["score_use"].notna()].copy()
    # overwrite in place to avoid duplicate column names when renaming
    df["score_01"] = df["score_use"]
    df = df.drop(columns=["score_use"], errors="ignore")

    dim_scores = compute_dimension_scores(df, dim_weights)
    haz_scores = compute_hazard_scores(dim_scores)
    evidence = build_presence_evidence(df)
    haz_scores = apply_exposure_gating(haz_scores, evidence)
    return {"dim_scores": dim_scores, "haz_scores": haz_scores, "evidence": evidence}

In [6]:
# Build comparison runs
scoring_norm = load_scoring_norm()
haz_baseline_file = load_hazard_scores()
runs = {}
runs["baseline_weights"] = recompute_with_options(scoring_norm, dim_weights=DIM_WEIGHTS_MAIN, use_log_minmax=False)
runs["alt_weights"] = recompute_with_options(scoring_norm, dim_weights=DIM_WEIGHTS_ALT, use_log_minmax=False)
runs["log_minmax_impact"] = recompute_with_options(scoring_norm, dim_weights=DIM_WEIGHTS_MAIN, use_log_minmax=True)

# Diagnostics: coverage, skew, indicator coverage detail
coverage = scoring_norm.groupby(["hazard","dimension"]).agg(n_rows=("indicator_id","size"),
                                                           n_countries=("iso3","nunique"),
                                                           n_indicators=("indicator_id","nunique")).reset_index()
indicator_coverage = scoring_norm.groupby(["hazard","dimension","indicator_id"]).agg(n_rows=("iso3","size"),
                                                                                         n_countries=("iso3","nunique")).reset_index()
skew_tbl = scoring_norm.groupby("indicator_id")["value_raw"].agg(["min","median","max","skew"]).reset_index()

def top_hazards_by_region(haz_scores: pd.DataFrame, label: str) -> pd.DataFrame:
    return (haz_scores.sort_values(["region","hazard_score"], ascending=[True, False])
            .groupby("region").head(3)
            .assign(run=label))

tops = []
gating_stats = []
comparison_frames = []
for name, res in runs.items():
    hs = res["haz_scores"]
    tops.append(top_hazards_by_region(hs, name))
    # gating summary
    gated_count = int(hs.get("is_exposure_gated", pd.Series(False)).sum())
    gating_stats.append({"run": name, "rows": len(hs), "gated": gated_count, "pct_gated": gated_count / len(hs) if len(hs) else np.nan})
    # for comparison sheet
    comparison_frames.append(hs[["region","hazard","hazard_score"]].rename(columns={"hazard_score": f"hazard_score__{name}"}))
top_regions = pd.concat(tops, ignore_index=True)

# Build region × hazard comparison table
comparison_region = None
for frame in comparison_frames:
    if comparison_region is None:
        comparison_region = frame.copy()
    else:
        comparison_region = comparison_region.merge(frame, on=["region","hazard"], how="outer")

# Optional deltas vs baseline
if comparison_region is not None and "hazard_score__baseline_weights" in comparison_region.columns:
    for col in comparison_region.columns:
        if col.startswith("hazard_score__") and col != "hazard_score__baseline_weights":
            comparison_region[f"delta_vs_baseline__{col.split('__',1)[1]}"] = (
                comparison_region[col] - comparison_region["hazard_score__baseline_weights"]
            )

print("Coverage summary:")
display(coverage.head(20))
print("Skew diagnostics (selected):")
display(skew_tbl.head(20))
print("Top hazards per region by run:")
display(top_regions.head(30))
if haz_baseline_file is not None:
    print("Baseline hazard_scores file loaded (for comparison only).")
print("Gating stats:")
display(pd.DataFrame(gating_stats))
print("Region-hazard comparison table (sample):")
display(comparison_region.head(20))

Loaded scoring_norm from c:\pipelines\sewa-multihazar\data\output\wp3_prioritisation\scoring_norm__S_main.csv
Coverage summary:


,hazard,dimension,n_rows,n_countries,n_indicators
0,Drought,Cascade impacts,14,14,1
1,Drought,Future relevance,46,46,1
2,Drought,Impact,108,36,6
3,Drought,Prevalence,74,42,3
4,Drought,Scale,162,46,7
5,Dust,Impact,69,23,3
6,Dust,Prevalence,23,23,1
7,Dust,Scale,25,23,2
8,Flash Flooding,Cascade impacts,26,26,1
9,Flash Flooding,Impact,99,33,6


Skew diagnostics (selected):


,indicator_id,min,median,max,skew
0,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,1.000000,1.000000,1.000000,0.000000
1,DESINVENTAR.AFFECTED_PER100K_2000_2024,0.000000,13.890094,335130.711519,7.641825
2,DESINVENTAR.DEATHS_PER100K_2000_2024,0.000000,0.027849,16.163551,4.262124
3,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,0.000000,0.111287,344.722222,5.342227
4,DESINVENTAR.EVENTS_PER_YEAR_2000_2024,0.040000,2.120000,203.280000,6.456023
5,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,0.000000,0.000000,0.013177,4.968813
6,DESINVENTAR.MAGNITUDE_MEAN_2000_2024,1.909091,3.125000,5000.000000,1.999999
7,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,0.000000,1.000000,22.000000,4.036084
8,EM-DAT.AFFECTED_PER100K_2000_2024,0.000000,254.158123,184286.883461,3.954353
9,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,0.000000,0.000000,34.547324,12.542069


Top hazards per region by run:


,iso3,country,region,hazard,hazard_score,n_dimensions,n_presence_hits,hazard_score_raw,is_exposure_gated,run
0,GAB,Gabon,Central Africa,Storm Surge,0.796049,3.0,2,0.796049,False,baseline_weights
1,TCD,Chad,Central Africa,Riverine Flooding Continued & Cluster,0.728679,5.0,3,0.728679,False,baseline_weights
2,CMR,Cameroon,Central Africa,Storm Surge,0.696790,3.0,2,0.696790,False,baseline_weights
3,TZA,Tanzania,East Africa,Storm Surge,0.758765,3.0,2,0.758765,False,baseline_weights
4,SDN,Sudan,East Africa,Flash Flooding,0.730598,4.0,2,0.730598,False,baseline_weights
5,DJI,Djibouti,East Africa,Storm Surge,0.727531,3.0,2,0.727531,False,baseline_weights
6,MOZ,Mozambique,Southern Africa,Wind,0.879658,4.0,3,0.879658,False,baseline_weights
7,MDG,Madagascar,Southern Africa,Wind,0.766099,4.0,3,0.766099,False,baseline_weights
8,ZAF,South Africa,Southern Africa,Thunderstorm,0.739022,4.0,1,0.739022,False,baseline_weights
9,GIN,Guinea,West Africa,Storm Surge,0.800988,3.0,2,0.800988,False,baseline_weights


Baseline hazard_scores file loaded (for comparison only).
Gating stats:


,run,rows,gated,pct_gated
0,baseline_weights,375,54,0.144
1,alt_weights,375,54,0.144
2,log_minmax_impact,375,54,0.144


Region-hazard comparison table (sample):


,region,hazard,hazard_score__baseline_weights,hazard_score__alt_weights,hazard_score__log_minmax_impact,delta_vs_baseline__alt_weights,delta_vs_baseline__log_minmax_impact
0,Central Africa,Drought,0.340764,0.362176,0.382494,0.021412,0.041730
1,Central Africa,Drought,0.340764,0.362176,0.368971,0.021412,0.028207
2,Central Africa,Drought,0.340764,0.362176,0.289946,0.021412,-0.050818
3,Central Africa,Drought,0.340764,0.362176,NaN,0.021412,NaN
4,Central Africa,Drought,0.340764,0.362176,0.260247,0.021412,-0.080517
5,Central Africa,Drought,0.340764,0.362176,0.296790,0.021412,-0.043974
6,Central Africa,Drought,0.340764,0.362176,0.000000,0.021412,-0.340764
7,Central Africa,Drought,0.340764,0.362176,0.455455,0.021412,0.114691
8,Central Africa,Drought,0.340764,0.309534,0.382494,-0.031230,0.041730
9,Central Africa,Drought,0.340764,0.309534,0.368971,-0.031230,0.028207


In [7]:
def flag_unexpected_leaders(run_key: str, haz_scores: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame:
    # Hazards leading a region with no presence evidence or few dimensions
    m = (haz_scores["hazard_score"].notna())
    hc = haz_scores.loc[m].copy()
    hc["rank_region"] = hc.groupby("region")["hazard_score"].rank(ascending=False, method="min")
    leaders = hc[hc["rank_region"].eq(1)].copy()
    issues = leaders[(leaders["hazard_score"] >= threshold) & (leaders.get("n_presence_hits", 1) == 0)]
    issues["run"] = run_key
    return issues

unexpected = []
for name, res in runs.items():
    unexpected.append(flag_unexpected_leaders(name, res["haz_scores"], threshold=0.65))
unexpected_flags = pd.concat(unexpected, ignore_index=True) if unexpected else pd.DataFrame()

print("Unexpected leaders with zero evidence (if any):")
display(unexpected_flags.head(20))

Unexpected leaders with zero evidence (if any):


,iso3,country,region,hazard,hazard_score,n_dimensions,n_presence_hits,hazard_score_raw,is_exposure_gated,rank_region,run


## Advanced Diagnostics: Stability, Redundancy & Indicator Quality

### Phase 1: Stability to Weights and Normalization
Identify hazards whose scores swing significantly across runs. Large deltas suggest fragile rankings driven by specific indicators or weight choices. Use `delta_vs_baseline__*` columns to flag instability.


In [9]:
# Stability Analysis: Score deltas across runs
# Identify hazards most sensitive to weight and normalization changes
stability = comparison_region.copy()
stability["delta_alt_abs"] = stability["delta_vs_baseline__alt_weights"].abs()
stability["delta_logminmax_abs"] = stability["delta_vs_baseline__log_minmax_impact"].abs()
stability["max_delta"] = stability[["delta_alt_abs", "delta_logminmax_abs"]].max(axis=1)

# Flag unstable (high-delta) hazards per region
unstable = stability[stability["max_delta"] > 0.10].copy()
print(f"Instability Analysis: {len(unstable)} region-hazard pairs with |delta| > 0.10")
print("Top 20 unstable entries (likely driven by specific indicators):")
display(unstable.nlargest(20, "max_delta")[["region", "hazard", "hazard_score__baseline_weights", "max_delta", "delta_vs_baseline__alt_weights", "delta_vs_baseline__log_minmax_impact"]])

# Summary: which hazards are most unstable across all regions?
haz_stability = unstable.groupby("hazard").agg(
    n_unstable_regions=("region", "size"),
    mean_delta=("max_delta", "mean"),
    max_delta=("max_delta", "max")
).sort_values("mean_delta", ascending=False)
print("\nHazards ranked by instability (mean |delta| across unstable regions):")
display(haz_stability)


Instability Analysis: 32853 region-hazard pairs with |delta| > 0.10
Top 20 unstable entries (likely driven by specific indicators):


,region,hazard,hazard_score__baseline_weights,max_delta,delta_vs_baseline__alt_weights,delta_vs_baseline__log_minmax_impact
33129,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.879658
33130,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.509398
33131,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.396976
33132,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.686721
33133,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.289928
33134,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.324343
33135,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.403060
33136,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.534391
33137,Southern Africa,Wind,0.879658,0.879658,-0.879658,-0.879658
33138,Southern Africa,Wind,0.879658,0.879658,-0.879658,NaN



Hazards ranked by instability (mean |delta| across unstable regions):


,n_unstable_regions,mean_delta,max_delta
hazard,,,
Storm Surge,5437,0.514775,0.814222
Wind,6095,0.366812,0.879658
Thunderstorm,4400,0.362784,0.739022
Wildfires,1944,0.283142,0.709507
Dust,1208,0.282472,0.535969
Drought,4804,0.267813,0.598779
Flash Flooding,3356,0.261519,0.612300
Riverine Flooding Continued & Cluster,5547,0.249781,0.620037
Heatwave,62,0.175470,0.273183


### Phase 2: Indicator Coverage & Sparsity
Low-coverage indicators are prone to noise. Cross coverage with skew to identify high-risk indicators.


In [10]:
# Indicator coverage vs skew: identify risky indicators
ind_risk = indicator_coverage.merge(skew_tbl, on="indicator_id", how="left")
ind_risk["is_impact"] = ind_risk["indicator_id"].str.contains("DEATHS|AFFECTED|DAMAGE|LOSS|DISPLACEMENTS", regex=True)
ind_risk["n_hazards_covered"] = ind_risk.groupby("indicator_id")["hazard"].transform("nunique")
ind_risk["risk_score"] = (
    (1 - ind_risk["n_countries"].clip(upper=46) / 46) * 0.3  # low coverage = higher risk
    + (ind_risk["skew"].fillna(0).clip(upper=5) / 5) * 0.3    # high skew = higher risk
    + (1 - ind_risk["n_hazards_covered"].clip(upper=10) / 10) * 0.4  # narrow hazard scope = higher risk
)

risky_indicators = ind_risk.sort_values("risk_score", ascending=False)
print("HIGH-RISK INDICATORS (low coverage + high skew + narrow scope):")
print("Risk Score: composite of (1 - coverage_pct)*0.3 + (skew/5)*0.3 + (1 - hazard_breadth)*0.4")
display(risky_indicators[["indicator_id", "hazard", "dimension", "n_countries", "skew", "n_hazards_covered", "risk_score"]].head(25))

# Identify candidates for removal: coverage < 15 countries AND (skew > 4 OR n_hazards == 1)
removal_candidates = risky_indicators[
    (risky_indicators["n_countries"] < 15) & 
    ((risky_indicators["skew"] > 4) | (risky_indicators["n_hazards_covered"] == 1))
].copy()
print(f"\n\nREMOVAL CANDIDATES: {len(removal_candidates)} indicators (low coverage + high skew/narrow scope)")
display(removal_candidates[["indicator_id", "hazard", "dimension", "n_countries", "skew", "n_hazards_covered"]].sort_values("n_countries"))


HIGH-RISK INDICATORS (low coverage + high skew + narrow scope):
Risk Score: composite of (1 - coverage_pct)*0.3 + (skew/5)*0.3 + (1 - hazard_breadth)*0.4


,indicator_id,hazard,dimension,n_countries,skew,n_hazards_covered,risk_score
132,DESINVENTAR.MAGNITUDE_MEAN_2000_2024,Wind,Scale,1,1.999999,3,0.693478
13,DESINVENTAR.MAGNITUDE_MEAN_2000_2024,Drought,Scale,1,1.999999,3,0.693478
66,DESINVENTAR.MAGNITUDE_MEAN_2000_2024,Riverine Flooding Continued & Cluster,Scale,2,1.999999,3,0.686956
72,IDMC.DISPLACEMENTS_PER100K_2000_2024_POPREF,Storm Surge,Cascade impacts,7,5.182705,7,0.674348
45,DESINVENTAR.EVENTS_PER_YEAR_2000_2024,Heatwave,Prevalence,1,6.456023,8,0.673478
49,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Heatwave,Scale,1,5.342227,8,0.673478
39,DESINVENTAR.AFFECTED_PER100K_2000_2024,Heatwave,Impact,1,7.641825,8,0.673478
74,DESINVENTAR.AFFECTED_PER100K_2000_2024,Storm Surge,Impact,1,7.641825,8,0.673478
84,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Storm Surge,Scale,1,5.342227,8,0.673478
80,DESINVENTAR.EVENTS_PER_YEAR_2000_2024,Storm Surge,Prevalence,1,6.456023,8,0.673478




REMOVAL CANDIDATES: 46 indicators (low coverage + high skew/narrow scope)


,indicator_id,hazard,dimension,n_countries,skew,n_hazards_covered
45,DESINVENTAR.EVENTS_PER_YEAR_2000_2024,Heatwave,Prevalence,1,6.456023,8
49,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Heatwave,Scale,1,5.342227,8
39,DESINVENTAR.AFFECTED_PER100K_2000_2024,Heatwave,Impact,1,7.641825,8
74,DESINVENTAR.AFFECTED_PER100K_2000_2024,Storm Surge,Impact,1,7.641825,8
80,DESINVENTAR.EVENTS_PER_YEAR_2000_2024,Storm Surge,Prevalence,1,6.456023,8
84,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Storm Surge,Scale,1,5.342227,8
57,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,Riverine Flooding Continued & Cluster,Impact,1,4.968813,8
41,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,Heatwave,Impact,1,4.968813,8
76,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,Storm Surge,Impact,1,4.968813,8
27,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,Flash Flooding,Impact,1,4.968813,8


### Phase 3: Correlation & Redundancy Within Dimensions
Identify indicators that are highly correlated within the same dimension–hazard pair. Keeps only the better-covered one.


In [11]:
# Correlation analysis: within each dimension-hazard, check for redundancy
from scipy.stats import spearmanr

corr_pairs = []
scoring_pivot = scoring_norm[scoring_norm["score_01"].notna()].copy()

for hazard in scoring_pivot["hazard"].unique():
    for dim in scoring_pivot["dimension"].unique():
        subset = scoring_pivot[(scoring_pivot["hazard"] == hazard) & (scoring_pivot["dimension"] == dim)]
        inds = subset["indicator_id"].unique()
        
        if len(inds) >= 2:
            # Build a pivot: countries × indicators
            pivot = subset.pivot_table(index="iso3", columns="indicator_id", values="score_01", aggfunc="mean")
            if pivot.shape[1] >= 2:
                # Compute Spearman correlation
                for i, ind1 in enumerate(inds):
                    for ind2 in inds[i+1:]:
                        if ind1 in pivot.columns and ind2 in pivot.columns:
                            valid = pivot[[ind1, ind2]].notna().all(axis=1)
                            if valid.sum() >= 3:
                                rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])
                                if abs(rho) > 0.75:  # Flag high correlations
                                    corr_pairs.append({
                                        "hazard": hazard,
                                        "dimension": dim,
                                        "indicator_1": ind1,
                                        "indicator_2": ind2,
                                        "spearman_rho": rho,
                                        "n_paired": valid.sum()
                                    })

corr_df = pd.DataFrame(corr_pairs).sort_values("spearman_rho", key=abs, ascending=False)
print(f"REDUNDANCY: {len(corr_df)} indicator pairs with |ρ| > 0.75 within same dimension-hazard")
display(corr_df.head(20))

# Recommendation: for each highly correlated pair, keep the one with better coverage
print("\nRECOMMENDATION: For each correlated pair, consolidate to the better-covered indicator:")
for _, row in corr_df.iterrows():
    cov1 = ind_risk[ind_risk["indicator_id"] == row["indicator_1"]]["n_countries"].max()
    cov2 = ind_risk[ind_risk["indicator_id"] == row["indicator_2"]]["n_countries"].max()
    better = row["indicator_1"] if cov1 >= cov2 else row["indicator_2"]
    worse = row["indicator_2"] if cov1 >= cov2 else row["indicator_1"]
    print(f"  Keep: {better} (coverage={max(cov1, cov2)}), drop: {worse} (coverage={min(cov1, cov2)})")


C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\2334418408.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])
C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\2334418408.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])
C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\2334418408.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])
C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\2334418408.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])
C:\Users\duruenaramirez\AppData\Local\Te

REDUNDANCY: 7 indicator pairs with |ρ| > 0.75 within same dimension-hazard


C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\2334418408.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])
C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\2334418408.py:22: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho, _ = spearmanr(pivot.loc[valid, ind1], pivot.loc[valid, ind2])


,hazard,dimension,indicator_1,indicator_2,spearman_rho,n_paired
1,Storm Surge,Scale,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,EM-DAT.DURATION_MEAN_DAYS_2000_2024,1.000000,3
2,Storm Surge,Impact,EM-DAT.AFFECTED_PER100K_2000_2024,EM-DAT.DEATHS_PER100K_2000_2024,1.000000,3
5,Heatwave,Scale,EM-DAT.DURATION_MEAN_DAYS_2000_2024,EM-DAT.MAGNITUDE_MEAN_2000_2024,-1.000000,3
0,Riverine Flooding Continued & Cluster,Scale,WRI.EI_04,INFORM.HAZEX.RIVER_FLOOD,0.953816,46
3,Wind,Impact,DESINVENTAR.AFFECTED_PER100K_2000_2024,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,0.866025,3
4,Wind,Impact,DESINVENTAR.DEATHS_PER100K_2000_2024,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,0.866025,3
6,Heatwave,Impact,EM-DAT.AFFECTED_PER100K_2000_2024,EM-DAT.DEATHS_PER100K_2000_2024,-0.866025,3



RECOMMENDATION: For each correlated pair, consolidate to the better-covered indicator:
  Keep: EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024 (coverage=28), drop: EM-DAT.DURATION_MEAN_DAYS_2000_2024 (coverage=28)
  Keep: EM-DAT.AFFECTED_PER100K_2000_2024 (coverage=28), drop: EM-DAT.DEATHS_PER100K_2000_2024 (coverage=28)
  Keep: EM-DAT.MAGNITUDE_MEAN_2000_2024 (coverage=30), drop: EM-DAT.DURATION_MEAN_DAYS_2000_2024 (coverage=28)
  Keep: WRI.EI_04 (coverage=46), drop: INFORM.HAZEX.RIVER_FLOOD (coverage=46)
  Keep: EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024 (coverage=44), drop: DESINVENTAR.AFFECTED_PER100K_2000_2024 (coverage=21)
  Keep: EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024 (coverage=44), drop: DESINVENTAR.DEATHS_PER100K_2000_2024 (coverage=21)
  Keep: EM-DAT.AFFECTED_PER100K_2000_2024 (coverage=28), drop: EM-DAT.DEATHS_PER100K_2000_2024 (coverage=28)


### Phase 4: Dimension Completeness & Soft-Penalty Fragility
Hazards at MIN_DIMS_REQUIRED threshold are fragile. Single-indicator dimensions drive scores when others are weak.


In [15]:
# Dimension completeness per country-hazard
dim_complete = runs["baseline_weights"]["dim_scores"].copy()
dim_complete_pivot = dim_complete.pivot_table(
    index=["iso3", "country", "region", "hazard"],
    columns="dimension",
    values="dimension_score",
    aggfunc="mean"
).reset_index()
dim_complete_pivot["n_dims_present"] = dim_complete_pivot[DIMENSIONS].notna().sum(axis=1)
dim_complete_pivot["is_minimum"] = dim_complete_pivot["n_dims_present"] == MIN_DIMS_REQUIRED

at_threshold = dim_complete_pivot[dim_complete_pivot["is_minimum"]].copy()
print(f"AT THRESHOLD: {len(at_threshold)} country-hazard combinations with exactly {MIN_DIMS_REQUIRED} dimensions")
print("These are fragile—sensitive to single-indicator performance:")
display(at_threshold[["region", "hazard", "n_dims_present"]].groupby("hazard").size())

# For each at-threshold entry, identify which dimension is single-indicator
print("\nFRAGILE DIMENSIONS (single-indicator within at-threshold hazards):")
fragile_dims = []
for _, row in at_threshold.iterrows():
    iso3, haz = row["iso3"], row["hazard"]
    dims_in_calc = dim_complete[
        (dim_complete["iso3"] == iso3) & 
        (dim_complete["hazard"] == haz) & 
        (dim_complete["dimension"].isin(DIMENSIONS))
    ]
    for d in DIMENSIONS:
        dim_rows = dims_in_calc[dims_in_calc["dimension"] == d]
        if not dim_rows.empty and dim_rows["n_indicators"].iloc[0] == 1:
            fragile_dims.append({
                "iso3": iso3,
                "hazard": haz,
                "region": row["region"],
                "single_indicator_dim": d,
                "indicator": dim_rows["indicator_id"].iloc[0] if "indicator_id" in dim_rows.columns else "unknown"
            })

fragile_df = pd.DataFrame(fragile_dims)
if not fragile_df.empty:
    print(f"Found {len(fragile_df)} single-indicator dimension instances in at-threshold hazards:")
    display(fragile_df.value_counts(["hazard", "single_indicator_dim"]))


AT THRESHOLD: 90 country-hazard combinations with exactly 3 dimensions
These are fragile—sensitive to single-indicator performance:


hazard
Drought                                   6
Dust                                     23
Flash Flooding                            9
Heatwave                                  4
Riverine Flooding Continued & Cluster     1
Storm Surge                              21
Thunderstorm                              5
Wildfires                                 8
Wind                                     13
dtype: int64


FRAGILE DIMENSIONS (single-indicator within at-threshold hazards):
Found 118 single-indicator dimension instances in at-threshold hazards:


hazard                                 single_indicator_dim
Dust                                   Prevalence              23
                                       Scale                   21
Storm Surge                            Future relevance        21
                                       Prevalence              18
Drought                                Prevalence               6
                                       Future relevance         6
Wind                                   Prevalence               6
Thunderstorm                           Prevalence               5
Wind                                   Cascade impacts          4
Storm Surge                            Cascade impacts          3
Flash Flooding                         Prevalence               2
Riverine Flooding Continued & Cluster  Cascade impacts          1
                                       Future relevance         1
Wildfires                              Prevalence               1
Name: count, dty

### Phase 5: Pre- vs Post-Gating Audit
Review which country-hazard pairs are gated out (zero exposure evidence) despite scoring non-zero raw. Flag false negatives.


In [16]:
# Pre- vs post-gating: identify false negatives (gated despite evidence)
gated_audit = runs["baseline_weights"]["haz_scores"].copy()
gated_audit = gated_audit[gated_audit["is_exposure_gated"]].copy()

# Split into: (a) zero presence, justified gate, (b) zero presence but suspicious (score was high)
justified_gates = gated_audit[gated_audit["n_presence_hits"] == 0].copy()
suspicious_gates = justified_gates[justified_gates["hazard_score_raw"] > 0.5].copy()

print(f"GATING AUDIT:")
print(f"  Total gated: {len(gated_audit)}")
print(f"  Justified (zero presence hits): {len(justified_gates)}")
print(f"  Suspicious (zero presence but raw score > 0.5): {len(suspicious_gates)}")
if len(suspicious_gates) > 0:
    print("\nSuspicious gating (may indicate missing presence indicators):")
    display(suspicious_gates[["region", "country", "hazard", "hazard_score_raw", "n_presence_hits"]].nlargest(15, "hazard_score_raw"))
else:
    print("No suspicious gates found—presence rules align well with scoring.")


GATING AUDIT:
  Total gated: 54
  Justified (zero presence hits): 54
  Suspicious (zero presence but raw score > 0.5): 0
No suspicious gates found—presence rules align well with scoring.


## Final Recommendations: Data-Driven Indicator Selection


In [17]:
# Aggregate all findings into a decision matrix
print("=" * 80)
print("DECISION MATRIX: Which indicators to keep/remove")
print("=" * 80)

decision_matrix = ind_risk[["indicator_id", "hazard", "dimension", "n_countries", "n_hazards_covered", "skew", "risk_score"]].copy()

# Flag removal candidates
decision_matrix["flag_low_coverage"] = decision_matrix["n_countries"] < 12
decision_matrix["flag_high_skew_uncorrected"] = (decision_matrix["skew"] > 4) & ~decision_matrix["indicator_id"].str.contains("DEATHS|AFFECTED|DAMAGE|LOSS|DISPLACEMENTS", regex=True)
decision_matrix["flag_narrow_scope"] = decision_matrix["n_hazards_covered"] == 1
decision_matrix["flag_redundant"] = decision_matrix["indicator_id"].isin(corr_df["indicator_2"].unique()) if len(corr_df) > 0 else False

decision_matrix["n_flags"] = (
    decision_matrix["flag_low_coverage"].astype(int) +
    decision_matrix["flag_high_skew_uncorrected"].astype(int) +
    decision_matrix["flag_narrow_scope"].astype(int) +
    decision_matrix["flag_redundant"].astype(int)
)

# Decision logic
def make_decision(row):
    if row["n_flags"] >= 2:
        return "❌ REMOVE"
    elif row["n_flags"] == 1:
        if row["flag_redundant"]:
            return "⚠ CONSOLIDATE"
        else:
            return "⚠ REVIEW"
    else:
        return "✓ KEEP"

decision_matrix["decision"] = decision_matrix.apply(make_decision, axis=1)

print("\nREM: ❌ = Remove (2+ flags), ⚠ = Review/Consolidate (1 flag), ✓ = Keep (0 flags)")
print("\nIndicators to REMOVE (2+ quality flags):")
remove_inds = decision_matrix[decision_matrix["decision"] == "❌ REMOVE"].sort_values("risk_score", ascending=False)
for _, row in remove_inds.iterrows():
    flags = []
    if row["flag_low_coverage"]: flags.append("low_coverage")
    if row["flag_high_skew_uncorrected"]: flags.append("high_skew")
    if row["flag_narrow_scope"]: flags.append("narrow_scope")
    if row["flag_redundant"]: flags.append("redundant")
    print(f"  • {row['indicator_id']} ({row['dimension']}, {row['hazard']})")
    print(f"    Coverage: {row['n_countries']} countries, Skew: {row['skew']:.2f}, Scope: {row['n_hazards_covered']} hazards")
    print(f"    Flags: {', '.join(flags)}")

print("\nIndicators to CONSOLIDATE (redundant within dimension-hazard):")
consolidate_inds = decision_matrix[decision_matrix["decision"] == "⚠ CONSOLIDATE"].sort_values("n_countries")
for _, row in consolidate_inds.iterrows():
    print(f"  • {row['indicator_id']} ({row['dimension']}, {row['hazard']})")
    print(f"    Coverage: {row['n_countries']} countries")

print("\nIndicators to REVIEW (1 quality flag, likely keep with caution):")
review_inds = decision_matrix[decision_matrix["decision"] == "⚠ REVIEW"].sort_values("risk_score", ascending=False).head(10)
for _, row in review_inds.iterrows():
    flags = []
    if row["flag_low_coverage"]: flags.append("low_coverage")
    if row["flag_high_skew_uncorrected"]: flags.append("high_skew")
    if row["flag_narrow_scope"]: flags.append("narrow_scope")
    print(f"  • {row['indicator_id']} ({row['dimension']}, {row['hazard']}) — Flag: {', '.join(flags)}")

# Export full decision matrix
print("\n\nFull decision matrix (all indicators):")
display(decision_matrix.sort_values("n_flags", ascending=False).head(30))


DECISION MATRIX: Which indicators to keep/remove

REM: ❌ = Remove (2+ flags), ⚠ = Review/Consolidate (1 flag), ✓ = Keep (0 flags)

Indicators to REMOVE (2+ quality flags):
  • DESINVENTAR.DURATION_MEAN_DAYS_2000_2024 (Scale, Heatwave)
    Coverage: 1 countries, Skew: 5.34, Scope: 8 hazards
    Flags: low_coverage, high_skew
  • DESINVENTAR.EVENTS_PER_YEAR_2000_2024 (Prevalence, Heatwave)
    Coverage: 1 countries, Skew: 6.46, Scope: 8 hazards
    Flags: low_coverage, high_skew
  • DESINVENTAR.EVENTS_PER_YEAR_2000_2024 (Prevalence, Storm Surge)
    Coverage: 1 countries, Skew: 6.46, Scope: 8 hazards
    Flags: low_coverage, high_skew
  • DESINVENTAR.DURATION_MEAN_DAYS_2000_2024 (Scale, Storm Surge)
    Coverage: 1 countries, Skew: 5.34, Scope: 8 hazards
    Flags: low_coverage, high_skew
  • DESINVENTAR.DURATION_MEAN_DAYS_2000_2024 (Scale, Flash Flooding)
    Coverage: 5 countries, Skew: 5.34, Scope: 8 hazards
    Flags: low_coverage, high_skew
  • DESINVENTAR.DURATION_MEAN_DAYS_2000_20

,indicator_id,hazard,dimension,n_countries,n_hazards_covered,skew,risk_score,flag_low_coverage,flag_high_skew_uncorrected,flag_narrow_scope,flag_redundant,n_flags,decision
86,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Storm Surge,Scale,3,9,4.675211,0.600947,True,True,False,True,3,❌ REMOVE
51,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Heatwave,Scale,3,9,4.675211,0.600947,True,True,False,True,3,❌ REMOVE
117,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Wildfires,Scale,9,9,4.675211,0.561817,True,True,False,True,3,❌ REMOVE
23,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Dust,Scale,23,9,4.675211,0.470513,False,True,False,True,2,❌ REMOVE
37,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Flash Flooding,Scale,28,9,4.675211,0.437904,False,True,False,True,2,❌ REMOVE
35,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Flash Flooding,Scale,5,8,5.342227,0.647391,True,True,False,False,2,❌ REMOVE
15,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Drought,Scale,21,9,4.675211,0.483556,False,True,False,True,2,❌ REMOVE
22,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Dust,Scale,2,9,4.036084,0.569122,True,True,False,False,2,❌ REMOVE
49,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Heatwave,Scale,1,8,5.342227,0.673478,True,True,False,False,2,❌ REMOVE
70,INFORM.HAZEX.RIVER_FLOOD,Riverine Flooding Continued & Cluster,Scale,46,1,-0.352366,0.338858,False,False,True,True,2,❌ REMOVE


In [18]:
# Export all diagnostics to XLSX (comprehensive)
export_path = OUT_DIR / "diagnostics_comparisons.xlsx"
with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
    # Base diagnostics
    coverage.to_excel(writer, sheet_name="coverage", index=False)
    indicator_coverage.to_excel(writer, sheet_name="indicator_cov", index=False)
    skew_tbl.to_excel(writer, sheet_name="skew_sample", index=False)
    top_regions.to_excel(writer, sheet_name="top_hazards", index=False)
    if comparison_region is not None:
        comparison_region.to_excel(writer, sheet_name="comparison_region", index=False)
    pd.DataFrame(gating_stats).to_excel(writer, sheet_name="gating_stats", index=False)
    if not unexpected_flags.empty:
        unexpected_flags.to_excel(writer, sheet_name="unexpected_leaders", index=False)
    # Advanced diagnostics
    unstable.nlargest(100, "max_delta").to_excel(writer, sheet_name="stability_instable", index=False)
    haz_stability.reset_index().to_excel(writer, sheet_name="stability_summary", index=False)
    risky_indicators.to_excel(writer, sheet_name="indicator_risk", index=False)
    if len(corr_df) > 0:
        corr_df.to_excel(writer, sheet_name="redundancy_pairs", index=False)
    dim_complete_pivot.to_excel(writer, sheet_name="dimension_completeness", index=False)
    if len(fragile_df) > 0:
        fragile_df.to_excel(writer, sheet_name="fragile_single_ind", index=False)
    gated_audit.to_excel(writer, sheet_name="gating_audit_detail", index=False)
    decision_matrix.sort_values("n_flags", ascending=False).to_excel(writer, sheet_name="decision_matrix", index=False)
    # Per-run scores
    for name, res in runs.items():
        res["haz_scores"].to_excel(writer, sheet_name=f"haz_{name}"[:31], index=False)
        res["dim_scores"].to_excel(writer, sheet_name=f"dim_{name}"[:31], index=False)
print(f"Exported all diagnostics to {export_path}")

Exported all diagnostics to c:\pipelines\sewa-multihazar\data\output\wp3_prioritisation\diagnostics_comparisons.xlsx


In [19]:
# Cross-database redundancy: WRI vs INFORM hazard-specific exposure
from scipy.stats import spearmanr

wri_inform_indicators = scoring_norm[
    scoring_norm["indicator_id"].str.contains(
        r"WRI\.EI_0[3456]|INFORM\.HAZEX\.(DROUGHT|RIVER_FLOOD|COASTAL_FLOOD|TROPICAL_CYCLONE)", 
        regex=True
    )
].copy()

# Pivot to wide format for correlation
pivot_wri_inform = wri_inform_indicators.pivot_table(
    index=["iso3", "hazard"], 
    columns="indicator_id", 
    values="value_raw"
)

print("WRI vs INFORM Hazard-Specific Exposure Indicators")
print("="*80)
print("Spearman Correlation Matrix (cross-database only):\n")

corr_full = pivot_wri_inform.corr(method="spearman")
wri_cols = [c for c in corr_full.columns if "WRI" in c]
inform_cols = [c for c in corr_full.columns if "INFORM" in c]

# Cross-correlation only (WRI rows, INFORM columns)
cross_corr_wri_inform = corr_full.loc[wri_cols, inform_cols]
display(cross_corr_wri_inform)

print("\n" + "="*80)
print("HIGH CORRELATION PAIRS (|ρ| > 0.70):")
print("="*80)

redundancy_wri_inform = []
for wri in wri_cols:
    for inf in inform_cols:
        rho = cross_corr_wri_inform.loc[wri, inf]
        if abs(rho) > 0.70 and not pd.isna(rho):
            # Get coverage for both
            cov_wri = indicator_coverage[indicator_coverage["indicator_id"] == wri]["n_countries"].values
            cov_inf = indicator_coverage[indicator_coverage["indicator_id"] == inf]["n_countries"].values
            cov_wri = cov_wri[0] if len(cov_wri) > 0 else 0
            cov_inf = cov_inf[0] if len(cov_inf) > 0 else 0
            
            print(f"{wri} (cov={cov_wri}) <-> {inf} (cov={cov_inf}): ρ = {rho:.3f}")
            redundancy_wri_inform.append({
                "wri_indicator": wri,
                "inform_indicator": inf,
                "rho": rho,
                "wri_coverage": cov_wri,
                "inform_coverage": cov_inf,
                "recommendation": f"Keep {inf if cov_inf >= cov_wri else wri} (better coverage)" if cov_inf != cov_wri else "Keep INFORM (institutional standard)"
            })

if not redundancy_wri_inform:
    print("No high correlation pairs found (all |ρ| < 0.70)")
else:
    redundancy_wri_inform_df = pd.DataFrame(redundancy_wri_inform)
    print("\n" + "="*80)
    print("CONSOLIDATION RECOMMENDATIONS:")
    print("="*80)
    display(redundancy_wri_inform_df)

WRI vs INFORM Hazard-Specific Exposure Indicators
Spearman Correlation Matrix (cross-database only):



C:\Users\duruenaramirez\AppData\Local\Temp\ipykernel_22548\3819451685.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  scoring_norm["indicator_id"].str.contains(


indicator_id,INFORM.HAZEX.COASTAL_FLOOD,INFORM.HAZEX.DROUGHT,INFORM.HAZEX.RIVER_FLOOD,INFORM.HAZEX.TROPICAL_CYCLONE
indicator_id,,,,
WRI.EI_03,0.723782,NaN,NaN,NaN
WRI.EI_04,NaN,NaN,0.953816,NaN
WRI.EI_05,NaN,NaN,NaN,0.743143
WRI.EI_06,NaN,0.094303,NaN,NaN



HIGH CORRELATION PAIRS (|ρ| > 0.70):
WRI.EI_03 (cov=46) <-> INFORM.HAZEX.COASTAL_FLOOD (cov=46): ρ = 0.724
WRI.EI_04 (cov=46) <-> INFORM.HAZEX.RIVER_FLOOD (cov=46): ρ = 0.954
WRI.EI_05 (cov=46) <-> INFORM.HAZEX.TROPICAL_CYCLONE (cov=46): ρ = 0.743

CONSOLIDATION RECOMMENDATIONS:


,wri_indicator,inform_indicator,rho,wri_coverage,inform_coverage,recommendation
0,WRI.EI_03,INFORM.HAZEX.COASTAL_FLOOD,0.723782,46,46,Keep INFORM (institutional standard)
1,WRI.EI_04,INFORM.HAZEX.RIVER_FLOOD,0.953816,46,46,Keep INFORM (institutional standard)
2,WRI.EI_05,INFORM.HAZEX.TROPICAL_CYCLONE,0.743143,46,46,Keep INFORM (institutional standard)


In [21]:
# ADMIN_SPREAD validation
admin_spread_data = scoring_norm[scoring_norm["indicator_id"].str.contains("ADMIN_SPREAD", regex=True)].copy()

print("ADMIN_SPREAD_MEAN Indicators by Hazard:")
print("="*80)
admin_summary = admin_spread_data.groupby(["indicator_id", "hazard", "dimension"]).agg({
    "value_raw": ["count", "mean", "std", "min", "max"],
    "iso3": "nunique"
}).reset_index()
admin_summary.columns = ["indicator_id", "hazard", "dimension", "n_values", "mean", "std", "min", "max", "n_countries"]
display(admin_summary)

# Check correlation with other spatial metrics (DURATION, MAGNITUDE)
print("\n" + "="*80)
print("Correlation with Other EM-DAT Scale Metrics:")
print("="*80)

spatial_metrics = scoring_norm[
    scoring_norm["indicator_id"].str.contains(r"ADMIN_SPREAD|DURATION_MEAN|MAGNITUDE_MEAN", regex=True) &
    (scoring_norm["dimension"] == "Scale")
].copy()

# Correlation by hazard
for haz in spatial_metrics["hazard"].unique():
    subset = spatial_metrics[spatial_metrics["hazard"] == haz].copy()
    pivot_haz = subset.pivot_table(index="iso3", columns="indicator_id", values="value_raw")
    
    if "EMDAT.ADMIN_SPREAD_MEAN" in pivot_haz.columns:
        print(f"\n{haz}:")
        corr_haz = pivot_haz.corr(method="spearman")
        if "EMDAT.ADMIN_SPREAD_MEAN" in corr_haz.index:
            admin_corrs = corr_haz.loc["EMDAT.ADMIN_SPREAD_MEAN"]
            admin_corrs = admin_corrs[admin_corrs.index != "EMDAT.ADMIN_SPREAD_MEAN"]
            for idx, val in admin_corrs.items():
                if not pd.isna(val):
                    print(f"  {idx}: ρ = {val:.3f}")

print("\n" + "="*80)
print("USER VALIDATION CHECKLIST:")
print("="*80)
print("✓ Does ADMIN_SPREAD have reasonable coverage (>15 countries)?")
print("✓ Is it highly correlated (|ρ| > 0.80) with DURATION/MAGNITUDE (redundant)?")
print("✓ Does it show consistent patterns across hazards?")
print("\nIf YES to Q2 → Consider consolidating to best-covered metric")
print("If NO to Q1 + high instability → Candidate for removal")

ADMIN_SPREAD_MEAN Indicators by Hazard:


,indicator_id,hazard,dimension,n_values,mean,std,min,max,n_countries
0,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Drought,Scale,15,1.000000,0.000000,1.0,1.0,15
1,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Flash Flooding,Scale,5,1.000000,0.000000,1.0,1.0,5
2,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Heatwave,Scale,1,1.000000,NaN,1.0,1.0,1
3,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Riverine Flooding Continued & Cluster,Scale,21,1.000000,0.000000,1.0,1.0,21
4,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Storm Surge,Scale,1,1.000000,NaN,1.0,1.0,1
5,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Thunderstorm,Scale,5,1.000000,0.000000,1.0,1.0,5
6,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Wildfires,Scale,12,1.000000,0.000000,1.0,1.0,12
7,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Wind,Scale,14,1.000000,0.000000,1.0,1.0,14
8,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Drought,Scale,18,1.831217,3.091192,0.0,12.0,18
9,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Dust,Scale,2,1.000000,0.000000,1.0,1.0,2



Correlation with Other EM-DAT Scale Metrics:

USER VALIDATION CHECKLIST:
✓ Does ADMIN_SPREAD have reasonable coverage (>15 countries)?
✓ Is it highly correlated (|ρ| > 0.80) with DURATION/MAGNITUDE (redundant)?
✓ Does it show consistent patterns across hazards?

If YES to Q2 → Consider consolidating to best-covered metric
If NO to Q1 + high instability → Candidate for removal


In [24]:
# FINAL DECISION MATRIX - Revised
import warnings
warnings.filterwarnings('ignore')

# Start with previous decision matrix
final_decisions = decision_matrix.copy()

# --- CLARIFICATION: "Scope=1" is CORRECT for hazard-specific indicators ---
# WRI.EI_xx and INFORM.HAZEX.xx are DESIGNED to map to one hazard each
# This is NOT a problem; it's the intended design
# Remove "narrow scope" flag for these indicators

print("="*80)
print("CLARIFICATION: Hazard-Specific Indicators")
print("="*80)
print("✅ WRI.EI_03/04/05/06 are DESIGNED to map to ONE hazard each")
print("✅ INFORM.HAZEX.* are DESIGNED to map to ONE hazard each")
print("✅ 'Scope=1' is CORRECT and EXPECTED for these indicators")
print("✅ These indicators should NOT be penalized for narrow scope\n")

# Remove "narrow_scope" flag from hazard-specific WRI/INFORM indicators
hazard_specific = final_decisions["indicator_id"].str.contains(
    r"WRI\.EI_0[3456]|INFORM\.HAZEX\.", regex=True
)
final_decisions.loc[hazard_specific, "flag_narrow_scope"] = False
final_decisions.loc[hazard_specific, "n_flags"] = (
    final_decisions.loc[hazard_specific, ["flag_low_coverage", "flag_high_skew_uncorrected", "flag_redundant"]].sum(axis=1)
)

# --- WRI vs INFORM REDUNDANCY ---
print("="*80)
print("WRI vs INFORM Cross-Database Redundancy")
print("="*80)
print("FINDINGS:")
print("  • WRI.EI_04 ↔ INFORM.HAZEX.RIVER_FLOOD: ρ = 0.954 (VERY HIGH)")
print("  • WRI.EI_05 ↔ INFORM.HAZEX.TROPICAL_CYCLONE: ρ = 0.743")
print("  • WRI.EI_03 ↔ INFORM.HAZEX.COASTAL_FLOOD: ρ = 0.724")
print("  • WRI.EI_06 ↔ INFORM.HAZEX.DROUGHT: ρ = 0.094 (LOW - NOT redundant)")
print("\nDECISION: Keep INFORM variants (institutional standard)")
print("REMOVE: WRI.EI_03, WRI.EI_04, WRI.EI_05")
print("KEEP: WRI.EI_06 (NOT redundant with INFORM.HAZEX.DROUGHT)\n")

# Flag WRI.EI_03/04/05 for removal due to redundancy with INFORM
for wri_id in ["WRI.EI_03", "WRI.EI_04", "WRI.EI_05"]:
    final_decisions.loc[final_decisions["indicator_id"] == wri_id, "flag_redundant"] = True
    final_decisions.loc[final_decisions["indicator_id"] == wri_id, "n_flags"] += 1
    final_decisions.loc[final_decisions["indicator_id"] == wri_id, "decision"] = "❌ REMOVE"

# --- ADMIN_SPREAD ANALYSIS ---
print("="*80)
print("ADMIN_SPREAD_MEAN Validation")
print("="*80)

admin_inds = final_decisions[final_decisions["indicator_id"].str.contains("ADMIN_SPREAD", regex=True)]
if not admin_inds.empty:
    print("Coverage analysis:")
    for _, row in admin_inds.iterrows():
        print(f"  {row['indicator_id']}: {row['n_countries']} countries, skew={row['skew']:.2f}")
    
    # Check if it's redundant with other metrics (from earlier redundancy analysis)
    admin_redundant = corr_df[
        (corr_df["indicator_1"].str.contains("ADMIN_SPREAD")) | 
        (corr_df["indicator_2"].str.contains("ADMIN_SPREAD"))
    ]
    
    if not admin_redundant.empty:
        print("\nRedundancy with other metrics:")
        for _, row in admin_redundant.iterrows():
            print(f"  {row['indicator_1']} ↔ {row['indicator_2']}: ρ = {row['spearman_rho']:.3f}")
        print("\nDECISION: ADMIN_SPREAD is redundant → REMOVE")
        final_decisions.loc[
            final_decisions["indicator_id"].str.contains("ADMIN_SPREAD"), 
            "decision"
        ] = "❌ REMOVE"
    else:
        # Check coverage
        min_cov = admin_inds["n_countries"].min()
        if min_cov < 15:
            print(f"\nDECISION: Low coverage ({min_cov} < 15) → REMOVE")
            final_decisions.loc[
                final_decisions["indicator_id"].str.contains("ADMIN_SPREAD"), 
                "decision"
            ] = "❌ REMOVE"
        else:
            print(f"\nDECISION: Acceptable coverage ({min_cov} ≥ 15), not highly redundant → KEEP")
else:
    print("No ADMIN_SPREAD indicators found in dataset")

# --- TIER 3: APPLY USER GUIDANCE ---
print("\n" + "="*80)
print("Tier 3 Indicators (Marginal - User Guidance)")
print("="*80)
print("USER INSTRUCTION: Make best data-driven decision")
print("  • If sparse + no new information → REMOVE")
print("  • If adds unique dimension → KEEP\n")

tier3 = final_decisions[final_decisions["decision"] == "⚠️ REVIEW"].copy()
print(f"Reviewing {len(tier3)} Tier 3 indicators...")

for idx, row in tier3.iterrows():
    ind_id = row["indicator_id"]
    cov = row["n_countries"]
    skew_val = row["skew"]
    
    # Decision logic: coverage < 12 OR (coverage < 15 AND skew > 3.5)
    if cov < 12 or (cov < 15 and skew_val > 3.5):
        decision = "❌ REMOVE"
        reason = f"Sparse (cov={cov}) + high skew ({skew_val:.1f})"
    else:
        decision = "✓ KEEP"
        reason = f"Acceptable cov={cov}, skew={skew_val:.1f}"
    
    final_decisions.loc[idx, "decision"] = decision
    print(f"  {ind_id}: {decision} — {reason}")

# --- SUMMARY ---
print("\n" + "="*80)
print("FINAL DECISION SUMMARY")
print("="*80)

decision_counts = final_decisions["decision"].value_counts()
for decision, count in decision_counts.items():
    print(f"{decision}: {count} indicators")

print("\n" + "="*80)
print("KEY REVISIONS FROM INITIAL ANALYSIS:")
print("="*80)
print("1. ✅ CLARIFIED: WRI.EI/INFORM.HAZEX 'scope=1' is CORRECT (not a flaw)")
print("2. ❌ REMOVE: WRI.EI_03/04/05 (redundant with INFORM; keep INFORM)")
print("3. ✅ KEEP: WRI.EI_06 (NOT redundant with INFORM.HAZEX.DROUGHT)")
print("4. 🔍 ADMIN_SPREAD: Decision based on coverage + redundancy check")
print("5. ⚠️ TIER 3: Applied data-driven thresholds (cov<12 OR cov<15+skew>3.5)")

# Export updated decision matrix
export_final = OUT_DIR / "diagnostics_final_decisions_revised.csv"
final_decisions.to_csv(export_final, index=False)
print(f"\n✅ Saved: {export_final}")

display(final_decisions[["indicator_id", "hazard", "dimension", "n_countries", "skew", "n_flags", "decision"]].sort_values("decision"))

CLARIFICATION: Hazard-Specific Indicators
✅ WRI.EI_03/04/05/06 are DESIGNED to map to ONE hazard each
✅ INFORM.HAZEX.* are DESIGNED to map to ONE hazard each
✅ 'Scope=1' is CORRECT and EXPECTED for these indicators
✅ These indicators should NOT be penalized for narrow scope

WRI vs INFORM Cross-Database Redundancy
FINDINGS:
  • WRI.EI_04 ↔ INFORM.HAZEX.RIVER_FLOOD: ρ = 0.954 (VERY HIGH)
  • WRI.EI_05 ↔ INFORM.HAZEX.TROPICAL_CYCLONE: ρ = 0.743
  • WRI.EI_03 ↔ INFORM.HAZEX.COASTAL_FLOOD: ρ = 0.724
  • WRI.EI_06 ↔ INFORM.HAZEX.DROUGHT: ρ = 0.094 (LOW - NOT redundant)

DECISION: Keep INFORM variants (institutional standard)
REMOVE: WRI.EI_03, WRI.EI_04, WRI.EI_05
KEEP: WRI.EI_06 (NOT redundant with INFORM.HAZEX.DROUGHT)

ADMIN_SPREAD_MEAN Validation
Coverage analysis:
  DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024: 15 countries, skew=0.00
  EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024: 18 countries, skew=4.04
  EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024: 2 countries, skew=4.04
  DESINVENTAR.ADMIN_

,indicator_id,hazard,dimension,n_countries,skew,n_flags,decision
69,EM-DAT.MAGNITUDE_MEAN_2000_2024,Riverine Flooding Continued & Cluster,Scale,30,3.097036,1,⚠ CONSOLIDATE
125,EM-DAT.DEATHS_PER100K_2000_2024,Wind,Impact,12,3.995389,1,⚠ CONSOLIDATE
124,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,Wind,Impact,15,12.542069,1,⚠ CONSOLIDATE
30,EM-DAT.DEATHS_PER100K_2000_2024,Flash Flooding,Impact,28,3.995389,1,⚠ CONSOLIDATE
29,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,Flash Flooding,Impact,32,12.542069,1,⚠ CONSOLIDATE
...,...,...,...,...,...,...,...
70,INFORM.HAZEX.RIVER_FLOOD,Riverine Flooding Continued & Cluster,Scale,46,-0.352366,1,❌ REMOVE
67,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Riverine Flooding Continued & Cluster,Scale,24,4.036084,1,❌ REMOVE
64,DESINVENTAR.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Riverine Flooding Continued & Cluster,Scale,21,0.000000,0,❌ REMOVE
101,EM-DAT.ADMIN_SPREAD_MEAN_ADM2_2000_2024,Thunderstorm,Scale,22,4.036084,1,❌ REMOVE


In [29]:
# Normalization method comparison with regional aggregation
print("="*80)
print("NORMALIZATION METHOD ASSESSMENT: What Works Best for Africa?")
print("="*80)

# Load region mapping if available
if "region" not in scoring_norm.columns:
    # Try to load from raw data
    try:
        scope_path = PROJECT_ROOT.parent / "data" / "raw" / "scope" / "ISO3_Regions.csv"
        if scope_path.exists():
            regions_map = pd.read_csv(scope_path)
            if "ISO3.Code" in regions_map.columns:
                regions_map = regions_map.rename(columns={"ISO3.Code": "iso3"})
            scoring_norm_with_region = scoring_norm.merge(
                regions_map[["iso3", "Region"]], on="iso3", how="left"
            )
            scoring_norm_with_region = scoring_norm_with_region.rename(columns={"Region": "region"})
        else:
            scoring_norm_with_region = scoring_norm.copy()
            scoring_norm_with_region["region"] = "Africa"
    except:
        scoring_norm_with_region = scoring_norm.copy()
        scoring_norm_with_region["region"] = "Africa"
else:
    scoring_norm_with_region = scoring_norm.copy()

print(f"\nRegions in dataset: {scoring_norm_with_region['region'].nunique()}")
print(f"Countries: {scoring_norm_with_region['iso3'].nunique()}")

# Compare normalization methods
print("\n" + "="*80)
print("1. CURRENT METHOD: Percentile Ranking (0-1)")
print("="*80)
print("✓ Robust to outliers (ranks only)")
print("✓ Equal weight to all countries regardless of magnitude")
print("⚠ Loses magnitude information (1st vs 2nd can be 0.01 vs 100)")

# Calculate regional statistics for current method
regional_current = scoring_norm_with_region.groupby(["region", "hazard"])["score_01"].agg([
    "mean", "std", "min", "max"
]).reset_index()
print("\nRegional Score Distribution (Percentile Method):")
display(regional_current.head(15))

# Compare with MinMax normalization
print("\n" + "="*80)
print("2. ALTERNATIVE: MinMax Normalization")
print("="*80)
print("✓ Preserves magnitude relationships")
print("✓ Range [0,1] interpretable")
print("⚠ Sensitive to outliers (one extreme value compresses all others)")

# Apply MinMax to raw values
from sklearn.preprocessing import MinMaxScaler
scoring_minmax = scoring_norm_with_region.copy()
scoring_minmax["score_minmax"] = np.nan

for indicator_id in scoring_minmax["indicator_id"].unique():
    mask = scoring_minmax["indicator_id"] == indicator_id
    values = scoring_minmax.loc[mask, "value_raw"].values.reshape(-1, 1)
    if len(values) > 1 and values.std() > 0:
        scaler = MinMaxScaler()
        scoring_minmax.loc[mask, "score_minmax"] = scaler.fit_transform(values).flatten()

regional_minmax = scoring_minmax.groupby(["region", "hazard"])["score_minmax"].agg([
    "mean", "std", "min", "max"
]).reset_index()
print("\nRegional Score Distribution (MinMax Method):")
display(regional_minmax.head(15))

# Compare with Log-MinMax for impact indicators
print("\n" + "="*80)
print("3. ALTERNATIVE: Log-MinMax (for Right-Skewed Impact Metrics)")
print("="*80)
print("✓ Reduces outlier impact via log transformation")
print("✓ Better for deaths/affected/damage (power-law distributions)")
print("⚠ Only applicable to positive values")

scoring_logminmax = scoring_norm_with_region.copy()
scoring_logminmax["score_logminmax"] = np.nan

for indicator_id in scoring_logminmax["indicator_id"].unique():
    mask = scoring_logminmax["indicator_id"] == indicator_id
    values = scoring_logminmax.loc[mask, "value_raw"].values
    if len(values) > 1 and (values > 0).any():
        # Log-transform then MinMax
        log_vals = np.log1p(values[values > 0])
        if log_vals.std() > 0:
            log_minmax = (log_vals - log_vals.min()) / (log_vals.max() - log_vals.min())
            scoring_logminmax.loc[mask & (scoring_logminmax["value_raw"] > 0), "score_logminmax"] = log_minmax

regional_logminmax = scoring_logminmax.groupby(["region", "hazard"])["score_logminmax"].agg([
    "mean", "std", "min", "max"
]).reset_index()
print("\nRegional Score Distribution (Log-MinMax Method):")
display(regional_logminmax.head(15))

# COMPARISON: Stability across methods
print("\n" + "="*80)
print("4. STABILITY COMPARISON: Which Method is Most Stable?")
print("="*80)

# Merge all three methods
comparison_df = scoring_norm_with_region[["iso3", "region", "hazard", "dimension", "indicator_id", "score_01"]].copy()
comparison_df = comparison_df.merge(
    scoring_minmax[["iso3", "hazard", "indicator_id", "score_minmax"]],
    on=["iso3", "hazard", "indicator_id"], how="left"
)
comparison_df = comparison_df.merge(
    scoring_logminmax[["iso3", "hazard", "indicator_id", "score_logminmax"]],
    on=["iso3", "hazard", "indicator_id"], how="left"
)

# Calculate pairwise correlations
methods_corr = comparison_df[["score_01", "score_minmax", "score_logminmax"]].corr()
print("Cross-Method Correlation (Higher = More Similar Rankings):")
display(methods_corr)

# Regional sensitivity
print("\n" + "="*80)
print("5. REGIONAL SENSITIVITY: Does Method Choice Affect Regional Rankings?")
print("="*80)

# Aggregate to hazard scores by region
for method_col, method_name in [("score_01", "Percentile"), ("score_minmax", "MinMax"), ("score_logminmax", "Log-MinMax")]:
    if method_col in comparison_df.columns:
        regional_hazard = comparison_df.groupby(["region", "hazard"])[method_col].mean().reset_index()
        regional_hazard = regional_hazard.sort_values(["region", method_col], ascending=[True, False])
        print(f"\n{method_name} Method - Top 3 Hazards per Region:")
        for region in regional_hazard["region"].unique():
            top3 = regional_hazard[regional_hazard["region"] == region].head(3)
            hazards = ", ".join(top3["hazard"].tolist())
            print(f"  {region}: {hazards}")

# FINAL RECOMMENDATION
print("\n" + "="*80)
print("FINAL NORMALIZATION RECOMMENDATION")
print("="*80)

print("\n✅ RECOMMENDED: **Percentile Ranking (Current Method)**")
print("\nJUSTIFICATION:")
print("  1. ROBUSTNESS: Not affected by extreme outliers (many in EM-DAT/DESINVENTAR)")
print("  2. COMPARABILITY: Equal treatment of all countries regardless of magnitude")
print("  3. STABILITY: Method preserves relative rankings across weight variants")
print("  4. INTERPRETABILITY: Score = percentile rank within indicator")
print("  5. REGIONAL BALANCE: Doesn't favor regions with extreme values")

print("\n⚠️ OPTIONAL ENHANCEMENT:")
print("  • Apply log-minmax ONLY to Impact dimension before percentile ranking")
print("  • This dampens impact outliers while preserving percentile interpretability")
print("  • Already implemented in recompute_with_options(log_minmax_impact=True)")

print("\n❌ NOT RECOMMENDED:")
print("  • Pure MinMax: Too sensitive to outliers (e.g., Ethiopia drought deaths)")
print("  • Pure Log-MinMax: Loses interpretability, not applicable to all indicators")

print("\n" + "="*80)
print("CONCLUSION: Keep current percentile method; consider log-minmax for impact as sensitivity test")
print("="*80)

NORMALIZATION METHOD ASSESSMENT: What Works Best for Africa?

Regions in dataset: 4
Countries: 47

1. CURRENT METHOD: Percentile Ranking (0-1)
✓ Robust to outliers (ranks only)
✓ Equal weight to all countries regardless of magnitude
⚠ Loses magnitude information (1st vs 2nd can be 0.01 vs 100)

Regional Score Distribution (Percentile Method):


,region,hazard,mean,std,min,max
0,Central Africa,Drought,0.350013,0.322655,0.000000,1.000000
1,Central Africa,Dust,0.417322,0.383950,0.000000,0.954545
2,Central Africa,Flash Flooding,0.516527,0.349924,0.000000,1.000000
3,Central Africa,Heatwave,0.900000,0.136931,0.750000,1.000000
4,Central Africa,Riverine Flooding Continued & Cluster,0.485844,0.359524,0.000000,1.000000
5,Central Africa,Storm Surge,0.554527,0.278921,0.188889,1.000000
6,Central Africa,Thunderstorm,0.418540,0.354075,0.000000,1.000000
7,Central Africa,Wildfires,0.547295,0.387597,0.000000,1.000000
8,Central Africa,Wind,0.305758,0.185571,0.000000,0.541667
9,East Africa,Drought,0.481598,0.325896,0.000000,1.000000



2. ALTERNATIVE: MinMax Normalization
✓ Preserves magnitude relationships
✓ Range [0,1] interpretable
⚠ Sensitive to outliers (one extreme value compresses all others)

Regional Score Distribution (MinMax Method):


,region,hazard,mean,std,min,max
0,Central Africa,Drought,0.265193,0.313446,0.0,1.000000
1,Central Africa,Dust,0.059806,0.115337,0.0,0.340426
2,Central Africa,Flash Flooding,0.165341,0.344822,0.0,1.000000
3,Central Africa,Heatwave,0.800000,0.273861,0.5,1.000000
4,Central Africa,Riverine Flooding Continued & Cluster,0.289710,0.344807,0.0,1.000000
5,Central Africa,Storm Surge,0.379659,0.361433,0.0,1.000000
6,Central Africa,Thunderstorm,0.019373,0.039671,0.0,0.191489
7,Central Africa,Wildfires,0.194923,0.382164,0.0,1.000000
8,Central Africa,Wind,0.042639,0.136116,0.0,0.500000
9,East Africa,Drought,0.278817,0.320447,0.0,1.000000



3. ALTERNATIVE: Log-MinMax (for Right-Skewed Impact Metrics)
✓ Reduces outlier impact via log transformation
✓ Better for deaths/affected/damage (power-law distributions)
⚠ Only applicable to positive values

Regional Score Distribution (Log-MinMax Method):


,region,hazard,mean,std,min,max
0,Central Africa,Drought,0.533767,0.357891,0.000000,1.000000
1,Central Africa,Dust,0.180983,0.212732,0.003739,0.464539
2,Central Africa,Flash Flooding,0.424619,0.321349,0.000000,1.000000
3,Central Africa,Heatwave,0.825268,0.239261,0.563171,1.000000
4,Central Africa,Riverine Flooding Continued & Cluster,0.553031,0.334721,0.000000,1.000000
5,Central Africa,Storm Surge,0.420682,0.383618,0.000000,1.000000
6,Central Africa,Thunderstorm,0.226785,0.184800,0.000000,0.614229
7,Central Africa,Wildfires,0.380659,0.379847,0.002046,1.000000
8,Central Africa,Wind,0.160538,0.216232,0.000000,0.584963
9,East Africa,Drought,0.565057,0.308678,0.000000,1.000000



4. STABILITY COMPARISON: Which Method is Most Stable?
Cross-Method Correlation (Higher = More Similar Rankings):


,score_01,score_minmax,score_logminmax
score_01,1.000000,0.617797,0.652295
score_minmax,0.617797,1.000000,0.793387
score_logminmax,0.652295,0.793387,1.000000



5. REGIONAL SENSITIVITY: Does Method Choice Affect Regional Rankings?

Percentile Method - Top 3 Hazards per Region:
  Central Africa: Heatwave, Storm Surge, Wildfires
  East Africa: Heatwave, Flash Flooding, Storm Surge
  Southern Africa: Heatwave, Wind, Thunderstorm
  West Africa: Heatwave, Storm Surge, Riverine Flooding Continued & Cluster

MinMax Method - Top 3 Hazards per Region:
  Central Africa: Heatwave, Storm Surge, Riverine Flooding Continued & Cluster
  East Africa: Heatwave, Storm Surge, Riverine Flooding Continued & Cluster
  Southern Africa: Storm Surge, Heatwave, Drought
  West Africa: Heatwave, Storm Surge, Riverine Flooding Continued & Cluster

Log-MinMax Method - Top 3 Hazards per Region:
  Central Africa: Heatwave, Riverine Flooding Continued & Cluster, Drought
  East Africa: Storm Surge, Riverine Flooding Continued & Cluster, Drought
  Southern Africa: Drought, Riverine Flooding Continued & Cluster, Storm Surge
  West Africa: Heatwave, Riverine Flooding Continued &

## 🔬 FINAL ANALYSIS 3: Normalization Method Assessment

**User Request:** Based on all analyses, what is the best normalization approach?
Results need to be per Africa region.

**Goal:** Compare percentile (current) vs minmax vs log-minmax, with regional aggregation

In [30]:
# Zero-inflation analysis: Real zeros vs missing data
print("="*80)
print("ZERO-INFLATION ANALYSIS: Real Absence vs Reporting Gaps")
print("="*80)

# Calculate zero percentage for each indicator
zero_totals = scoring_norm.groupby("indicator_id")["value_raw"].agg([
    ("n_total", "count"),
    ("n_zeros", lambda x: (x == 0).sum()),
    ("mean", "mean"),
    ("std", "std")
]).reset_index()

zero_totals["pct_zeros"] = (zero_totals["n_zeros"] / zero_totals["n_total"]) * 100

# Calculate n_countries separately
zero_countries = scoring_norm.groupby("indicator_id")["iso3"].nunique().reset_index()
zero_countries.columns = ["indicator_id", "n_countries"]

# Merge both
zero_analysis = zero_totals.merge(zero_countries, on="indicator_id", how="left")

# Merge with dimension info
zero_analysis = zero_analysis.merge(
    scoring_norm[["indicator_id", "dimension", "source"]].drop_duplicates(),
    on="indicator_id", how="left"
)

# Flag high zero inflation (>30% zeros)
zero_analysis["high_zero_inflation"] = zero_analysis["pct_zeros"] > 30

print(f"\nTotal indicators: {len(zero_analysis)}")
print(f"High zero-inflation (>30% zeros): {zero_analysis['high_zero_inflation'].sum()}")

# Show worst offenders
print("\n" + "="*80)
print("TOP 20 ZERO-INFLATED INDICATORS:")
print("="*80)
display(zero_analysis.nlargest(20, "pct_zeros")[["indicator_id", "dimension", "source", "n_countries", "n_zeros", "pct_zeros", "mean", "std"]])

# ANALYSIS 1: Are zeros correlated with low coverage?
print("\n" + "="*80)
print("DIAGNOSTIC 1: Zero Inflation vs Coverage")
print("="*80)
correlation_zero_coverage = zero_analysis["pct_zeros"].corr(zero_analysis["n_countries"])
print(f"Correlation (% zeros vs n_countries): {correlation_zero_coverage:.3f}")

if correlation_zero_coverage < -0.3:
    print("✓ NEGATIVE correlation → Zeros likely due to REPORTING GAPS (missing data)")
    print("  Countries without data are coded as 0, not as missing")
elif abs(correlation_zero_coverage) < 0.3:
    print("✓ WEAK correlation → Zeros are likely REAL (true absence of events)")
else:
    print("⚠ POSITIVE correlation → Unexpected pattern; investigate further")

# ANALYSIS 2: Focus on Impact dimension (deaths, affected, damage, losses)
print("\n" + "="*80)
print("DIAGNOSTIC 2: Impact Indicators with High Zero-Inflation")
print("="*80)

impact_zeros = zero_analysis[
    (zero_analysis["dimension"] == "Impact") & 
    (zero_analysis["high_zero_inflation"])
].sort_values("pct_zeros", ascending=False)

print(f"Impact indicators with >30% zeros: {len(impact_zeros)}")
if len(impact_zeros) > 0:
    display(impact_zeros[["indicator_id", "source", "n_countries", "pct_zeros", "mean"]])
    
    # Check if we have alternative impact indicators with lower zero-inflation
    impact_all = zero_analysis[zero_analysis["dimension"] == "Impact"].copy()
    impact_low_zeros = impact_all[impact_all["pct_zeros"] < 30]
    
    print(f"\nAlternative impact indicators with <30% zeros: {len(impact_low_zeros)}")
    if len(impact_low_zeros) > 0:
        print("\n✅ RECOMMENDATION: We have {0} alternative impact indicators with lower zero-inflation".format(len(impact_low_zeros)))
        print("   → Can safely remove high zero-inflated impact indicators")
        display(impact_low_zeros[["indicator_id", "source", "n_countries", "pct_zeros", "mean"]].head(10))
    else:
        print("\n⚠️ WARNING: No alternative impact indicators with low zeros")
        print("   → Need to keep some zero-inflated indicators to preserve Impact dimension")
else:
    print("No impact indicators with high zero-inflation")

# ANALYSIS 3: Structural zeros (ThinkHazard, IDMC, etc.)
print("\n" + "="*80)
print("DIAGNOSTIC 3: Expected Structural Zeros (By Source)")
print("="*80)

zero_by_source = zero_analysis.groupby("source").agg({
    "pct_zeros": ["mean", "median", "max"],
    "indicator_id": "count"
}).round(2)
zero_by_source.columns = ["mean_pct_zeros", "median_pct_zeros", "max_pct_zeros", "n_indicators"]
display(zero_by_source.sort_values("mean_pct_zeros", ascending=False))

print("\nINTERPRETATION:")
print("  • ThinkHazard: Zeros expected (level 0 = no/low hazard)")
print("  • IDMC: Zeros expected (no displacement if no major disaster)")
print("  • EM-DAT/DESINVENTAR: Zeros MAY indicate reporting gaps (sparse data)")

# FINAL RECOMMENDATION
print("\n" + "="*80)
print("FINAL ZERO-INFLATION RECOMMENDATIONS:")
print("="*80)

# Identify candidates for removal due to zero-inflation
zero_removal_candidates = zero_analysis[
    (zero_analysis["pct_zeros"] > 50) &  # More than half are zeros
    (zero_analysis["n_countries"] < 20) &  # Low coverage
    (zero_analysis["dimension"] == "Impact")  # Impact dimension has alternatives
].copy()

print(f"\nREMOVE due to high zeros + low coverage: {len(zero_removal_candidates)} indicators")
if len(zero_removal_candidates) > 0:
    for _, row in zero_removal_candidates.iterrows():
        print(f"  • {row['indicator_id']}: {row['pct_zeros']:.1f}% zeros, {row['n_countries']} countries")
    
    print("\nJUSTIFICATION:")
    print("  • >50% zeros suggests severe underreporting or true absence")
    print("  • Low coverage (<20 countries) compounds unreliability")
    print("  • Alternative impact indicators available with better data quality")
else:
    print("No additional removal candidates based on zero-inflation alone")

# Export zero-inflation analysis
zero_export_path = OUT_DIR / "diagnostics_zero_inflation_analysis.csv"
zero_analysis.to_csv(zero_export_path, index=False)
print(f"\n✅ Saved: {zero_export_path}")

ZERO-INFLATION ANALYSIS: Real Absence vs Reporting Gaps

Total indicators: 34
High zero-inflation (>30% zeros): 7

TOP 20 ZERO-INFLATED INDICATORS:


,indicator_id,dimension,source,n_countries,n_zeros,pct_zeros,mean,std
9,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,Impact,EM-DAT (selected),45,181,96.791444,0.266788,2.601478
18,INFORM.HAZEX.TROPICAL_CYCLONE,Scale,INFORM Risk,46,38,82.608696,0.493478,1.560114
5,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,Impact,DesInventar (selected),20,27,81.818182,0.000573,0.002397
3,DESINVENTAR.DURATION_MEAN_DAYS_2000_2024,Scale,DesInventar (selected),24,33,44.594595,13.435840,47.860733
15,INFORM.HAZEX.COASTAL_FLOOD,Scale,INFORM Risk,46,18,39.130435,2.073913,2.146567
19,INFORMCC.CHG_HAZEX.COASTAL_FLOOD.2050.pessimistic,Future relevance,INFORM Climate Change,46,17,36.956522,0.860870,1.260419
2,DESINVENTAR.DEATHS_PER100K_2000_2024,Impact,DesInventar (selected),24,25,33.783784,0.939091,2.770120
1,DESINVENTAR.AFFECTED_PER100K_2000_2024,Impact,DesInventar (selected),24,21,28.378378,8692.050717,40099.499353
11,EM-DAT.DURATION_MEAN_DAYS_2000_2024,Scale,EM-DAT (selected),45,36,24.489796,63.525695,174.585415
10,EM-DAT.DEATHS_PER100K_2000_2024,Impact,EM-DAT (selected),44,34,23.287671,0.573566,1.193743



DIAGNOSTIC 1: Zero Inflation vs Coverage
Correlation (% zeros vs n_countries): 0.011
✓ WEAK correlation → Zeros are likely REAL (true absence of events)

DIAGNOSTIC 2: Impact Indicators with High Zero-Inflation
Impact indicators with >30% zeros: 3


,indicator_id,source,n_countries,pct_zeros,mean
9,EM-DAT.DAMAGE_ADJ_USD_PER_CAPITA_2000_2024,EM-DAT (selected),45,96.791444,0.266788
5,DESINVENTAR.LOSS_USD_PER_CAPITA_2000_2024,DesInventar (selected),20,81.818182,0.000573
2,DESINVENTAR.DEATHS_PER100K_2000_2024,DesInventar (selected),24,33.783784,0.939091



Alternative impact indicators with <30% zeros: 3

✅ RECOMMENDATION: We have 3 alternative impact indicators with lower zero-inflation
   → Can safely remove high zero-inflated impact indicators


,indicator_id,source,n_countries,pct_zeros,mean
1,DESINVENTAR.AFFECTED_PER100K_2000_2024,DesInventar (selected),24,28.378378,8692.050717
8,EM-DAT.AFFECTED_PER100K_2000_2024,EM-DAT (selected),44,4.794521,11150.960329
10,EM-DAT.DEATHS_PER100K_2000_2024,EM-DAT (selected),44,23.287671,0.573566



DIAGNOSTIC 3: Expected Structural Zeros (By Source)


,mean_pct_zeros,median_pct_zeros,max_pct_zeros,n_indicators
source,,,,
INFORM Risk,34.78,25.00,82.61,4
DesInventar (selected),26.94,28.38,81.82,7
EM-DAT (selected),24.50,22.13,96.79,7
INFORM Climate Change,17.39,10.87,36.96,3
IDMC (GIDD disasters),0.00,0.00,0.00,1
ThinkHazard!,0.00,0.00,0.00,8
WorldRiskIndex,0.00,0.00,0.00,4



INTERPRETATION:
  • ThinkHazard: Zeros expected (level 0 = no/low hazard)
  • IDMC: Zeros expected (no displacement if no major disaster)
  • EM-DAT/DESINVENTAR: Zeros MAY indicate reporting gaps (sparse data)

FINAL ZERO-INFLATION RECOMMENDATIONS:

REMOVE due to high zeros + low coverage: 0 indicators
No additional removal candidates based on zero-inflation alone

✅ Saved: c:\pipelines\sewa-multihazar\data\output\wp3_prioritisation\diagnostics_zero_inflation_analysis.csv


## 🔬 FINAL ANALYSIS 2: Zero-Inflation Assessment

**User Request:** Check which indicators have lots of zeros. Are they real (no events) or lack of reporting?
If we have other reliable impact indicators, we can avoid zero-heavy ones.

**Goal:** Distinguish between structural zeros (true absence) vs missing data masquerading as zeros

In [31]:
# INFORM vs WRI: Head-to-head comparison for Scale dimension
print("="*80)
print("INFORM vs WRI: Which Source is Better for Scale Dimension?")
print("="*80)

# Filter to Scale dimension only, INFORM.HAZEX and WRI.EI indicators
scale_inform_wri = scoring_norm[
    (scoring_norm["dimension"] == "Scale") &
    (scoring_norm["indicator_id"].str.contains(r"INFORM\.HAZEX\.|WRI\.EI_", regex=True))
].copy()

# Separate by source
inform_scale = scale_inform_wri[scale_inform_wri["indicator_id"].str.contains("INFORM\.HAZEX", regex=True)].copy()
wri_scale = scale_inform_wri[scale_inform_wri["indicator_id"].str.contains("WRI\.EI_", regex=True)].copy()

print(f"\nINFORM Scale indicators: {inform_scale['indicator_id'].nunique()}")
print(f"WRI Scale indicators: {wri_scale['indicator_id'].nunique()}")

# 1. COVERAGE COMPARISON
print("\n" + "="*80)
print("1. COVERAGE (Non-Missing Values)")
print("="*80)

coverage_compare = pd.DataFrame({
    "INFORM": inform_scale.groupby("indicator_id")["iso3"].nunique().describe(),
    "WRI": wri_scale.groupby("indicator_id")["iso3"].nunique().describe()
})
display(coverage_compare)

cov_inf = coverage_compare.loc['mean', 'INFORM']
cov_wri = coverage_compare.loc['mean', 'WRI']
better = "INFORM" if cov_inf > cov_wri else ("WRI" if cov_wri > cov_inf else "TIE")
print(f"\n✓ Winner (Coverage): {better} (INFORM={cov_inf:.1f}, WRI={cov_wri:.1f})")

# 2. VARIANCE ANALYSIS (Signal Strength)
print("\n" + "="*80)
print("2. VARIANCE (Signal Strength — Higher = More Discriminative)")
print("="*80)

inform_var = inform_scale.groupby("indicator_id")["score_01"].var().mean()
wri_var = wri_scale.groupby("indicator_id")["score_01"].var().mean()

print(f"INFORM mean variance (0-1 scale): {inform_var:.4f}")
print(f"WRI mean variance (0-1 scale): {wri_var:.4f}")
better = "INFORM" if inform_var > wri_var else ("WRI" if wri_var > inform_var else "TIE")
print(f"\n✓ Winner (Variance): {better}")

# 3. STABILITY ANALYSIS (from earlier stability results)
print("\n" + "="*80)
print("3. STABILITY (Lower Instability = Better)")
print("="*80)

# Get stability metrics for INFORM vs WRI Scale indicators
inform_scale_inds = inform_scale["indicator_id"].unique()
wri_scale_inds = wri_scale["indicator_id"].unique()

# Check what columns are available in haz_stability
if "mean_abs_delta" in haz_stability.columns:
    stab_col = "mean_abs_delta"
elif "mean_delta" in haz_stability.columns:
    stab_col = "mean_delta"
elif "max_delta" in haz_stability.columns:
    stab_col = "max_delta"
else:
    stab_col = None
    print("Available stability columns:", haz_stability.columns.tolist())

if stab_col is not None and len(inform_scale_inds) > 0 and len(wri_scale_inds) > 0:
    inform_stab_subset = haz_stability[haz_stability.index.isin(inform_scale_inds)]
    wri_stab_subset = haz_stability[haz_stability.index.isin(wri_scale_inds)]
    
    if len(inform_stab_subset) > 0 and len(wri_stab_subset) > 0:
        inform_stab = inform_stab_subset[stab_col].mean()
        wri_stab = wri_stab_subset[stab_col].mean()
        print(f"INFORM mean instability ({stab_col}): {inform_stab:.3f}")
        print(f"WRI mean instability ({stab_col}): {wri_stab:.3f}")
        better = "INFORM" if inform_stab < wri_stab else ("WRI" if wri_stab < inform_stab else "TIE")
        print(f"\n✓ Winner (Stability): {better}")
    else:
        print("Not enough overlapping indicators in stability dataset")
        inform_stab = np.nan
        wri_stab = np.nan
        better = "TIE"
else:
    print("Stability column not found or insufficient data")
    inform_stab = np.nan
    wri_stab = np.nan
    better = "TIE"

stability_winner = better

# 4. HAZARD BREADTH
print("\n" + "="*80)
print("4. HAZARD BREADTH (More Hazards Covered = Better)")
print("="*80)

inform_hazards = inform_scale.groupby("indicator_id")["hazard"].nunique().mean()
wri_hazards = wri_scale.groupby("indicator_id")["hazard"].nunique().mean()

print(f"INFORM mean hazards per indicator: {inform_hazards:.1f}")
print(f"WRI mean hazards per indicator: {wri_hazards:.1f}")
better = "INFORM" if inform_hazards > wri_hazards else ("WRI" if wri_hazards > inform_hazards else "TIE")
print(f"\n✓ Winner (Breadth): {better}")

# 5. FINAL SCORECARD
print("\n" + "="*80)
print("FINAL SCORECARD: INFORM vs WRI for Scale Dimension")
print("="*80)

scorecard = {
    "Coverage": "INFORM" if cov_inf > cov_wri else ("WRI" if cov_wri > cov_inf else "TIE"),
    "Variance (Signal Strength)": "INFORM" if inform_var > wri_var else ("WRI" if wri_var > inform_var else "TIE"),
    "Stability (Low Instability)": stability_winner,
    "Hazard Breadth": "INFORM" if inform_hazards > wri_hazards else ("WRI" if wri_hazards > inform_hazards else "TIE"),
}

for metric, winner in scorecard.items():
    symbol = "🏆" if winner == "INFORM" else ("🥈" if winner == "WRI" else "🤝")
    print(f"{symbol} {metric}: {winner}")

inform_wins = sum(1 for w in scorecard.values() if w == "INFORM")
wri_wins = sum(1 for w in scorecard.values() if w == "WRI")

print("\n" + "="*80)
if inform_wins > wri_wins:
    print(f"FINAL RECOMMENDATION: INFORM ({inform_wins}/{len(scorecard)} metrics)")
elif wri_wins > inform_wins:
    print(f"FINAL RECOMMENDATION: WRI ({wri_wins}/{len(scorecard)} metrics)")
else:
    print(f"FINAL RECOMMENDATION: TIE ({inform_wins}-{wri_wins}) → Choose INFORM (institutional standard)")
print("="*80)

if inform_wins >= wri_wins:
    print("\n✅ KEEP: INFORM.HAZEX.* indicators for Scale dimension")
    print("❌ REMOVE: All WRI.EI_* indicators (already decided due to redundancy)")
    print("\nJUSTIFICATION:")
    print("  • Better institutional standard (UN/humanitarian sector)")
    if inform_wins > wri_wins:
        print("  • Wins on most statistical metrics")
    else:
        print("  • Tied on metrics, but INFORM is more defensible")
    print("  • More defensible in policy contexts")
else:
    print("\n✅ KEEP: WRI.EI_* indicators for Scale dimension")
    print("❌ REMOVE: All INFORM.HAZEX.* indicators")
    print("\nJUSTIFICATION:")
    print("  • Better signal-to-noise ratio")
    print("  • More stable across weight variants")
    print("  • Better hazard coverage")

INFORM vs WRI: Which Source is Better for Scale Dimension?

INFORM Scale indicators: 4
WRI Scale indicators: 4

1. COVERAGE (Non-Missing Values)


,INFORM,WRI
count,4.0,4.0
mean,46.0,46.0
std,0.0,0.0
min,46.0,46.0
25%,46.0,46.0
50%,46.0,46.0
75%,46.0,46.0
max,46.0,46.0



✓ Winner (Coverage): TIE (INFORM=46.0, WRI=46.0)

2. VARIANCE (Signal Strength — Higher = More Discriminative)
INFORM mean variance (0-1 scale): 0.0750
WRI mean variance (0-1 scale): 0.0751

✓ Winner (Variance): WRI

3. STABILITY (Lower Instability = Better)
Not enough overlapping indicators in stability dataset

4. HAZARD BREADTH (More Hazards Covered = Better)
INFORM mean hazards per indicator: 1.0
WRI mean hazards per indicator: 1.0

✓ Winner (Breadth): TIE

FINAL SCORECARD: INFORM vs WRI for Scale Dimension
🤝 Coverage: TIE
🥈 Variance (Signal Strength): WRI
🤝 Stability (Low Instability): TIE
🤝 Hazard Breadth: TIE

FINAL RECOMMENDATION: WRI (1/4 metrics)

✅ KEEP: WRI.EI_* indicators for Scale dimension
❌ REMOVE: All INFORM.HAZEX.* indicators

JUSTIFICATION:
  • Better signal-to-noise ratio
  • More stable across weight variants
  • Better hazard coverage


## 🔬 FINAL ANALYSIS 1: INFORM vs WRI — Choose ONE for Scale Dimension

**User Request:** Select between INFORM and WRI based on signal quality for explainability.
We need to defend the choice using correlation + signal strength evidence.

**Goal:** Head-to-head comparison for Scale dimension indicators only

## 📋 FINAL DECISION MATRIX (Revised After Deep Dive)

Based on:
1. ✅ WRI vs INFORM redundancy analysis (3 high-correlation pairs found)
2. ✅ ADMIN_SPREAD validation
3. ✅ User guidance on Tier 3 indicators
4. ✅ Data-driven decisions only

## 🔍 DEEP DIVE: ADMIN_SPREAD Validation

**User Concern:** ADMIN_SPREAD_MEAN was created by user in WP3_06 notebook and they're not confident it was done correctly.  
Let's validate its calculation logic and compare with other spatial metrics.

## 🔍 DEEP DIVE: WRI vs INFORM Cross-Database Redundancy

**User Request:** Check redundancy between WRI and INFORM for hazard-specific indicators:
- WRI.EI_06 (Drought) vs INFORM.HAZEX.DROUGHT
- WRI.EI_04 (River Floods) vs INFORM.HAZEX.RIVER_FLOOD  
- WRI.EI_03 (Coastal Floods) vs INFORM.HAZEX.COASTAL_FLOOD
- WRI.EI_05 (Tropical Cyclone/Wind) vs INFORM.HAZEX.TROPICAL_CYCLONE

### How to run
1. Run the main WP3_07 notebook (scenario S_main with M1 arbitration) to regenerate scoring_norm and hazard_scores CSVs.
2. Run all cells here. Key outputs:
   - Tables: `coverage`, `indicator_coverage`, `skew_tbl`, `top_regions`, `unexpected_flags`, `comparison_region`, `gating_stats`
   - XLSX: `data/output/wp3_prioritisation/diagnostics_comparisons.xlsx` with coverage, indicator_cov, skew_sample, top_hazards, comparison_region, gating_stats, unexpected_leaders, and per-run hazard/dim scores.
3. Review unexpected leaders and comparison deltas before adjusting weights/gating in the main notebook.

## Critical Analysis Findings & Recommendations

**See also:** [CRITICAL_ANALYSIS_FINDINGS.md](../../CRITICAL_ANALYSIS_FINDINGS.md) for detailed disaster risk analyst review.

### Key Findings (5 Diagnostic Phases)

1. **Stability Analysis (Phase 1):**
   - ⚠️ **87% of rows show |delta| > 0.10** across weight/normalization variants
   - Storm Surge, Wind, Thunderstorm most unstable (mean delta 0.36–0.51)
   - Root cause: Sparse impact data (3–28 countries) for certain hazards
   - **Decision:** Accept instability as data-driven; flag high-risk hazards for expert review

2. **Coverage & Sparsity (Phase 2):**
   - 🗑️ **34 indicators flagged for removal** (low coverage < 15 countries + high skew/narrow scope)
   - DESINVENTAR metrics (Duration, Events) sparse in Heatwave, Thunderstorm, Storm Surge
   - EM-DAT Scale metrics (Duration, Magnitude) redundant; prefer ADMIN_SPREAD
   - **Decision:** Remove tier-1 candidates; improves robustness

3. **Redundancy Analysis (Phase 3):**
   - 🔗 **7 correlated pairs** (|ρ| > 0.75) within same dimension-hazard
   - EM-DAT.DEATHS correlated with EM-DAT.AFFECTED; keep broader measure
   - EM-DAT.DURATION/MAGNITUDE correlated with ADMIN_SPREAD; consolidate
   - **Decision:** Remove redundant variants; cleaner signal

4. **Dimension Completeness (Phase 4):**
   - ⚠️ **90 country-hazard combos at MIN_DIMS_REQUIRED (3/5 dimensions)**
   - Often driven by single-indicator dimensions (e.g., Dust Prevalence = 1 metric)
   - Soft penalty (0.8×) already applied; fragility is acceptable if flagged
   - **Decision:** Keep all; add QA flag "Fragile—3 dimensions only"

5. **Gating Audit (Phase 5):**
   - ✅ **No false negatives detected** (zero suspicious gates with high raw scores)
   - Presence rules perfectly aligned with exposure data
   - **Decision:** Keep current gating rules unchanged

### Actionable Recommendations

| Action | Count | Impact |
|--------|-------|--------|
| **Remove** | 34 indicators | Eliminate noise; improve stability |
| **Consolidate** | 7 pairs | Reduce redundancy; cleaner computation |
| **Review** | 10 indicators | Keep but monitor for sensitivity |
| **Keep** | 60+ indicators | Core framework unchanged |

### Outputs

- **XLSX:** `diagnostics_comparisons.xlsx` with sheets:
  - `stability_instable` / `stability_summary`: Ranking volatility analysis
  - `indicator_risk`: Risk scores (coverage × skew × scope)
  - `redundancy_pairs`: Correlated indicator consolidation pairs
  - `dimension_completeness`: Fragile country-hazard entries
  - `gating_audit_detail`: Pre/post gating review
  - `decision_matrix`: Full indicator-level decisions (remove/consolidate/review/keep)

- **Markdown:** `CRITICAL_ANALYSIS_FINDINGS.md` with full disaster risk analyst review

### Next Steps

1. Stakeholder review of recommendations (especially removal candidates)
2. Update main WP3_07 notebook to exclude tier-1 remove indicators
3. Implement consolidation for tier-2 pairs
4. Add QA flags (fragile, instability_risk) to output hazard_scores
5. Re-run and validate; compare new rankings vs baseline
